# Report figures from verified experiment artifacts

This notebook produces the thesis's **data charts and experimental image panels** from extracted Results folders or Kaggle notebook chaining inputs. Manually maintained conceptual and architecture diagrams are excluded; their originals and Draw.io sources remain in the report. All plotting code is embedded here: no `thesis_common.py`, repo checkout, internet access, package installation, model download, or training is needed for the default run.

**The report is not edited.** Outputs are written into a separate folder. Each numerical chart is reconstructed from saved logits, metrics or logged history, with item counts and evaluation provenance. The design follows the corresponding PNG in `Report/figures`: chart type, panel arrangement, model-family colors and task labels are preserved while numerical values are updated. Every source is SHA-256 hashed in `generation_manifest.json`. **PNG is the only figure format**, with new charts rendered at 600 dpi. Source data and provenance are exported as CSV/JSON.

## Kaggle notebook chaining

1. Create a new Kaggle notebook, then use **Add Input → Notebook Output Files** to attach the completed producers. For the complete supplied collection, attach SC1, SC2, SC3, SC4, SC5, SC6, SC7, SC9, SC10, SC10b, SC12, SC13, SC18, SC21, SC22, SC23, SC24, SC25, SC26 and SC30. SC14/SC17 cached figures are unnecessary because this notebook recomputes the benchmark. Attach the separate `Report_Figures` artifact to retain historical detector curves; it is not used for numerical classification results. SC20 can replace historical SC9 only after the checks described below.
2. Run the configuration cell. With `PREFER_DIRECT_NOTEBOOK_INPUTS=True`, a producer individually attached at `/kaggle/input/notebooks/<owner>/<producer-slug>` takes precedence over captured copies inside `all-output`. Ownership uses the first mount boundary, so nested copies do not masquerade as individual attachments. This is an explicit input-scope policy, not a claim that a file is newer or from the correct dataset. Selected candidates and ignored snapshots are recorded in `discovery_events`. If there are multiple directly attached versions, byte-identical copies are verified and collapsed; differing versions require `SC_ROOT_OVERRIDES`. Explicit overrides always take precedence. If no matching direct attachment exists, normal discovery and conflict checks apply to bundled inputs. Set the preference to `False` to compare all discovered copies instead. SC7 and SC26 deliberately remain distinct despite sharing output filenames. Missing companion files are never borrowed from captured nested notebook inputs.
3. Run all cells. Download `/kaggle/working/report_figures`. Start review with `figure_status.csv`, `README.md` and `generation_manifest.json`. Missing evidence or conflicting optional raster versions create actionable manifest entries while the other figures continue. Set `FAIL_ON_MISSING=True` to raise only after all available exports and the complete status manifest have been written.
4. Inspect the generation manifest and review captions before copying selected figures to the report. Check the dataset release, image versus item evaluation units, and calibration ownership.

Required plotting packages (`numpy`, `pandas`, `matplotlib`, `scikit-learn`, `Pillow`) are already present in standard Kaggle Python images. `torch` is additionally used to safely read checkpoint tensor counts for the partial efficiency figure. If PyTorch is absent, that section is marked missing while the other figures continue. If a required plotting import fails, choose an environment with those packages; this notebook does not install dependencies automatically.

## Interpretation rules

* SC1/SC12 and the comparable newer classifiers have 485-item development data and 50 held-out test items. SC19 and the supplied SC9 OOF files belong to the older 457-item release. SC19's 46-item test is never put into the current 50-item benchmark.
* The SC9 **source defaults to three folds**, but the saved OOF archive does not include fold identifiers or executed-run metadata. Figures therefore say **Pooled out-of-fold** with the actual item count, without a fold-count claim. No completed SC20 five-fold/485-item outputs were supplied.
* General threat accuracy/QWK/confusion use `sum(cumulative CORN probability > .5)`, matching `full_report`. SC15 pooling analysis uses the source notebook's **class-probability argmax**. Its threat≥2 recall is a head metric, not the rate of hazardous items routed to Special.
* SC6 temperatures belong to **SC7 fusion**, established by reproducing all four before/after ECE values. They must not be transplanted to SC12, SC26 or SC9. The supplied 3-vector threat temperature differs from the current source's scalar-temperature recipe; this notebook preserves and identifies the actual artifact.
* SC6 chooses winners using test scores. These are exploratory comparisons. This plotting notebook does not adopt a new deployment model based on test rankings.
* Existing detection, Grad-CAM and qualitative PNGs lack sufficient raw coordinates/images/CAM arrays for faithful visual redesign. They are honestly marked `legacy_raster` and retain the supplied PNG pixels. The SC22 showcase includes only **item_001, item_101 and item_049**, in that order. Updated SC22 exports `qualitative_manifest.json`, which verifies the three-item PNG before it is copied. The exact supplied historical four-row PNG can also be filtered safely by its recorded SHA-256 identity: only the item_218 row is removed. Any other PNG without matching showcase metadata is refused. Qualitative exports still require provenance review, especially SC22's incompatible historical calibration.
* Efficiency latency measurements and current detector raw curves are absent. The default efficiency figure derives actual checkpoint state-tensor counts and head-score differences, explicitly marking latency as pending. Optional explicit-input detector reevaluation below can compute real current curves.

## Missing data that can be derived

Condition confusion and empirical ROC curves are derived directly from the real OOF logits. SC15's seven parameter-free aggregation rules are also reproduced directly from saved per-view logits, so no completed SC15 run is required. No learned attention/DeepSets result or new training claim is invented. Conceptual workflow and architecture artwork is maintained separately from these data plots.

`CV_SOURCE="auto"` prefers a future completed SC20 export only when all five per-fold test files are present, disjoint and exactly reproduce the pooled logits, the count matches SC1, and its labels agree with SC12's held-out items. These checks establish test-fold coverage and artifact consistency; training/validation separation and full dataset identity additionally require the upstream manifests/run fingerprint. An incomplete or inconsistent SC20 never silently replaces SC9. Set `CV_SOURCE="SC9"` explicitly to retain the historical analysis.

## Figure labels

Visible titles, axes, ticks, legends, and annotations use model names, task names, and evaluation descriptions. Notebook identifiers are retained only in input configuration, exported data, and provenance manifests. Every newly plotted figure is checked before export to prevent notebook identifiers from appearing in its labels. Existing raster panels retain their supplied pixels.

## Scope: 28 PNG charts and image panels

The following manual report diagrams are intentionally excluded; update their report/Draw.io artwork separately:

- `intro_overview.png`
- `End_to_End_Inference.png`
- `Flow_Chart.png`
- `pipeline_architecture.png`
- `Dataset_Development.png`
- `Labelling_Scheme.png`
- `Preprocessing.png`
- `yolo11_architecture_abstract.png`
- `convnext_s_architecture_abstract.png`
- `decision_layer.png`

Five have verified Draw.io sources; the other five are excluded conceptual/architecture artwork. The generation manifest records this distinction and the hashes of every inspected report style reference.


In [ ]:
# Set an extracted Results root explicitly only when auto-detection cannot find it.
LOCAL_RESULTS_ROOT = ""

# Prefer an individually attached producer over captured copies inside all-output.
# Applies to the observed notebooks/<owner>/<slug> mount layout. Set False to
# require equality across direct and bundled copies. Explicit overrides win.
PREFER_DIRECT_NOTEBOOK_INPUTS = True

# Each override must point at exactly one producer/version's artifact directory.
# Kaggle examples (replace with the actual mounts shown in your Inputs panel):
# SC_ROOT_OVERRIDES = {
#     "SC7": "/kaggle/input/notebooks/yourname/sc7-fusion-train-eval",
#     "SC26": "/kaggle/input/notebooks/yourname/sc26-fusion-vitb",
#     "SC12": "/kaggle/input/notebooks/yourname/sc12-train-convnext-small-in22k",
#     "SC21": "/kaggle/input/notebooks/yourname/sc21-gradcam",
# }
SC_ROOT_OVERRIDES = {}

# Main deployment reference stays explicit; no auto-selection using test performance.
PRIMARY_SC = "SC12"
CV_SOURCE = "auto"          # Prefer verified complete SC20 if attached; otherwise SC9, with an explicit discovery event.
DIAGNOSTIC_SC = "cv"        # Selected OOF pool; "SC9", "SC20" or "SC12" explicitly select another evaluation.
VIEW_AGGREGATION_SC = "cv"  # Selected OOF pool; "SC12" derives a separate 50-item experiment.
INCLUDE_BASELINE_FUSION = False # Default benchmark = nine single backbones + SC26; SC7 is a separate baseline.
QUALITATIVE_SHOWCASE_ITEMS = ["item_001", "item_101", "item_049"]
QUALITATIVE_EXCLUDED_ITEMS = ["item_218"] # Presentation selection only; full test predictions are preserved.
OUTPUT_DIR = ""             # Default: /kaggle/working/report_figures or project/Generated_Report_Figures.
SHOW_FIGURES = True
FAIL_ON_MISSING = False      # True raises after writing all available figures and the complete status manifest.

# Optional actual efficiency measurements. CSV must contain:
# source,params_M,infer_ms,device,batch_size,input_size,precision,preprocessing_included
# Use unique rows for SC12 and SC7/SC26, same hardware, precision, batch size,
# warmup and synchronized CUDA timings. No estimated latency is accepted.
CLASSIFIER_EFFICIENCY_CSV = ""

# Optional real detector reevaluation. Dependencies must already be installed.
# Explicit paths avoid accidental selection of another notebook's best.pt or TACO data.
RUN_DETECTOR_REEVALUATION = False
DETECTOR_CHECKPOINT = ""
DETECTOR_DATA_YAML = ""   # Must declare the intended current test split; reviewed manually against 50 items/221 views.
DETECTOR_DEVICE = "0"     # GPU index on Kaggle; set "cpu" when needed.

ALL_REPORT_REFERENCES = ['Dataset_Development.png', 'End_to_End_Inference.png', 'Flow_Chart.png', 'Labelling_Scheme.png', 'Preprocessing.png', 'calibration_reliability.png', 'cls_compare_condition_auroc.png', 'cls_compare_material_f1.png', 'cls_compare_route_f1.png', 'cls_compare_threat_qwk.png', 'condition_roc.png', 'confusion_condition.png', 'confusion_material.png', 'confusion_route.png', 'confusion_threat.png', 'convnext_s_architecture_abstract.png', 'convnext_s_training_curves.png', 'decision_layer.png', 'detection_F1.png', 'detection_P.png', 'detection_PR.png', 'detection_R.png', 'efficiency_tradeoff.png', 'end_to_end.png', 'external_validation.png', 'fig1_labelling_transparent.png', 'fig2_preprocessing_transparent.png', 'fig3_detector_classifier_transparent.png', 'gradcam_grid.png', 'intro_overview.png', 'kfold_ci.png', 'material_distribution.png', 'pipeline_architecture.png', 'qualitative_demo.png', 'route_roc.png', 'route_threat_distribution.png', 'view_aggregation_plot.png', 'yolo11_architecture_abstract.png']
EXCLUDED_REPORT_FIGURES = [{'figure': 'intro_overview.png', 'reason': 'Matching maintained DrawIo page and embedded PNG mxfile; user excluded maintained diagrams.', 'exclusion_basis': 'maintained_drawio_source', 'drawio_source': 'Report/DrawIo/intro_overview.drawio'}, {'figure': 'End_to_End_Inference.png', 'reason': 'Conceptual/architecture asset excluded under the chosen chart/photo-only generator scope; no maintained DrawIo source was found.', 'exclusion_basis': 'manual_chart_photo_only_scope'}, {'figure': 'Flow_Chart.png', 'reason': 'Conceptual/architecture asset excluded under the chosen chart/photo-only generator scope; no maintained DrawIo source was found.', 'exclusion_basis': 'manual_chart_photo_only_scope'}, {'figure': 'pipeline_architecture.png', 'reason': 'Matching maintained DrawIo page and embedded PNG mxfile; user excluded maintained diagrams.', 'exclusion_basis': 'maintained_drawio_source', 'drawio_source': 'Report/DrawIo/pipeline_architecture.drawio'}, {'figure': 'Dataset_Development.png', 'reason': 'Conceptual/architecture asset excluded under the chosen chart/photo-only generator scope; no maintained DrawIo source was found.', 'exclusion_basis': 'manual_chart_photo_only_scope'}, {'figure': 'Labelling_Scheme.png', 'reason': 'Matching maintained DrawIo page and embedded PNG mxfile; user excluded maintained diagrams.', 'exclusion_basis': 'maintained_drawio_source', 'drawio_source': 'Report/DrawIo/Labelling_Scheme.drawio'}, {'figure': 'Preprocessing.png', 'reason': 'Conceptual/architecture asset excluded under the chosen chart/photo-only generator scope; no maintained DrawIo source was found.', 'exclusion_basis': 'manual_chart_photo_only_scope'}, {'figure': 'yolo11_architecture_abstract.png', 'reason': 'Conceptual/architecture asset excluded under the chosen chart/photo-only generator scope; no maintained DrawIo source was found.', 'exclusion_basis': 'manual_chart_photo_only_scope'}, {'figure': 'convnext_s_architecture_abstract.png', 'reason': 'Matching maintained DrawIo page and embedded PNG mxfile; user excluded maintained diagrams.', 'exclusion_basis': 'maintained_drawio_source', 'drawio_source': 'Report/DrawIo/convnext_s_architecture_abstract.drawio'}, {'figure': 'decision_layer.png', 'reason': 'Matching maintained DrawIo page and embedded PNG mxfile; user excluded maintained diagrams.', 'exclusion_basis': 'maintained_drawio_source', 'drawio_source': 'Report/DrawIo/decision_layer.drawio'}]
REPORT_REFERENCES = ['calibration_reliability.png', 'cls_compare_condition_auroc.png', 'cls_compare_material_f1.png', 'cls_compare_route_f1.png', 'cls_compare_threat_qwk.png', 'condition_roc.png', 'confusion_condition.png', 'confusion_material.png', 'confusion_route.png', 'confusion_threat.png', 'convnext_s_training_curves.png', 'detection_F1.png', 'detection_P.png', 'detection_PR.png', 'detection_R.png', 'efficiency_tradeoff.png', 'end_to_end.png', 'external_validation.png', 'fig1_labelling_transparent.png', 'fig2_preprocessing_transparent.png', 'fig3_detector_classifier_transparent.png', 'gradcam_grid.png', 'kfold_ci.png', 'material_distribution.png', 'qualitative_demo.png', 'route_roc.png', 'route_threat_distribution.png', 'view_aggregation_plot.png']

# Original report PNGs were inspected for design only; Kaggle does not load their old numerical content.
STYLE_REFERENCES = {'calibration_reliability.png': {'path': 'Report/figures/calibration_reliability.png', 'sha256': '0480482f3b06e57e14cb3e58ccdb4586fcef8bf1266c3f4469eb4533974e3c09'}, 'cls_compare_condition_auroc.png': {'path': 'Report/figures/cls_compare_condition_auroc.png', 'sha256': '8d0e4718639e200fd9febdf726d8d02cd64236f9479db355faae31ca9292c704'}, 'cls_compare_material_f1.png': {'path': 'Report/figures/cls_compare_material_f1.png', 'sha256': 'ac68c2a285f88752822a8dbf68e486ff3508ef03e87e2c5fc5d9eb4b03dd7b46'}, 'cls_compare_route_f1.png': {'path': 'Report/figures/cls_compare_route_f1.png', 'sha256': 'f69552dd54ba8d6285425b0e40ba3a9399ae27e0c098194bc2f926525d2d3b53'}, 'cls_compare_threat_qwk.png': {'path': 'Report/figures/cls_compare_threat_qwk.png', 'sha256': '010bad77619245d2428b27c7c1c0e4970f6dfb1342b73450afbd72f77545a427'}, 'condition_roc.png': {'path': 'Report/figures/condition_roc.png', 'sha256': '7d3ddabf6dbf31e158ef34f1b4b7302464e583d35779e31f6d557a0534efdeb7'}, 'confusion_condition.png': {'path': 'Report/figures/confusion_condition.png', 'sha256': '569c68975059259f3b9dc2c4cff45b844268cb59fb727feda9f64457875a72d4'}, 'confusion_material.png': {'path': 'Report/figures/confusion_material.png', 'sha256': '30e124fba10fcf1e7542f197a3bf545d7aac886f488bdba93e5c73a4511929e2'}, 'confusion_route.png': {'path': 'Report/figures/confusion_route.png', 'sha256': 'a3566e5622d93ee368603aadf9751e0d781998d420ff81927665d51487387ee1'}, 'confusion_threat.png': {'path': 'Report/figures/confusion_threat.png', 'sha256': '87963feaf2818c64de910c35a3c8e11d448a847ef5264dbd1c73a22ba5c58d1e'}, 'convnext_s_training_curves.png': {'path': 'Report/figures/convnext_s_training_curves.png', 'sha256': 'a7cce7aa2c823185bf850912884b18243f85fae41be570e34b4001cd500d2f5c'}, 'detection_F1.png': {'path': 'Report/figures/detection_F1.png', 'sha256': '11d170d39948d0d2d04f36eff2d05ab5b51368d8fa3b6a0a79ca7a7ff0dd4181'}, 'detection_P.png': {'path': 'Report/figures/detection_P.png', 'sha256': 'fff262d36669ba7254c8bb3efd8afd70f1da5c9aa1fa94701236fc8971f7da6b'}, 'detection_PR.png': {'path': 'Report/figures/detection_PR.png', 'sha256': '3b9bda463b67aa5d7070a448f97b20331047293c61301a1f593af90815992c48'}, 'detection_R.png': {'path': 'Report/figures/detection_R.png', 'sha256': 'd480196a1863c85b458bad3149c5d9f611fbf54763eec71ce5eb9b4fd1d7230f'}, 'efficiency_tradeoff.png': {'path': 'Report/figures/efficiency_tradeoff.png', 'sha256': '52a914aecb2e61f2d0a68830c65df0d97e3d5939a6e5d9fa07fb82547f7bde43'}, 'end_to_end.png': {'path': 'Report/figures/end_to_end.png', 'sha256': '3cbd6bbe72ed2ae2629a0ab5297e191be806c9ff228e454beb754b31080e6277'}, 'external_validation.png': {'path': 'Report/figures/external_validation.png', 'sha256': '7a45f1b36758c5104096952969d7a29de775c5e2d43d894ea3a0d1c296a2dffb'}, 'fig1_labelling_transparent.png': {'path': 'Report/figures/fig1_labelling_transparent.png', 'sha256': '6c50bfd14d89e48ef00c315fc80844e16425463d01054bbb0757dfd04be37f2f'}, 'fig2_preprocessing_transparent.png': {'path': 'Report/figures/fig2_preprocessing_transparent.png', 'sha256': 'a506970fc4c6988ebed85b31025e0722616083b06148a20726cbef5034d65d1a'}, 'fig3_detector_classifier_transparent.png': {'path': 'Report/figures/fig3_detector_classifier_transparent.png', 'sha256': 'b94f1d7efaf37e653a8cc920e74a4228b340540a3a55de7d20f3a00bbc7aa5b7'}, 'gradcam_grid.png': {'path': 'Report/figures/gradcam_grid.png', 'sha256': '4df4f62eb253a3b15922bf093dc6834e493c30892d7509f914ff05e4f7b4b661'}, 'kfold_ci.png': {'path': 'Report/figures/kfold_ci.png', 'sha256': '49d522b7fed6f2f37d6d83ae1b9769e216900e33c5c48cc68b0af213ed997eb1'}, 'material_distribution.png': {'path': 'Report/figures/material_distribution.png', 'sha256': 'a19a1cefdb2d6d95d4585c1fecab7a6a90a13109cd49500c582b03134639983f'}, 'qualitative_demo.png': {'path': 'Report/figures/qualitative_demo.png', 'sha256': 'c1c8971d1866b44f36d8cb56e4191b498d1d2a404d24d50986840a65c4a72eed'}, 'route_roc.png': {'path': 'Report/figures/route_roc.png', 'sha256': 'c8422f2d88d08262e195258e4b1fea9f380a012700d0a96d37b4797ecacedbf4'}, 'route_threat_distribution.png': {'path': 'Report/figures/route_threat_distribution.png', 'sha256': 'dd6e4051cc1a11b6006449d2859147daa96cbac1e4c6351ad6e8954291817bc8'}, 'view_aggregation_plot.png': {'path': 'Report/figures/view_aggregation_plot.png', 'sha256': '822271806b4470b30773058d9ee2c965a0539aac6544a139d56e980285c20567'}}


## Provenance and input discovery

In [ ]:
from pathlib import Path
from collections import defaultdict
import os, re, json, hashlib, shutil, textwrap, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.metrics import (confusion_matrix, f1_score, roc_curve, roc_auc_score,
                             cohen_kappa_score, accuracy_score)
from PIL import Image

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 12,
    'axes.titleweight': 'normal', 'axes.labelsize': 10, 'axes.edgecolor': '#B7BEC4',
    'axes.labelcolor': '#30363B', 'text.color': '#242A2F', 'xtick.color': '#40484F',
    'ytick.color': '#40484F', 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'axes.prop_cycle': plt.cycler(color=['#315F8C','#00838F','#E65100','#C2185B','#2E7D32','#90A4AE']),
})
INK, BLUE, TEAL, ORANGE, RED, GRAY = '#242A2F','#315F8C','#00838F','#E65100','#C2185B','#90A4AE'
GREEN = '#2E7D32'
ROUTE_COLORS = ['#78909C','#00897B','#3A6EA5','#C2185B']
PALETTE = [BLUE, TEAL, ORANGE, '#8559A5', RED, '#466E75', '#A58134', '#427E9C', '#8F557B', GRAY]
HEADS = ['material', 'condition', 'route', 'threat']
MATERIALS = ['plastic','paper','metal','glass','textile','e_waste','battery','sharps','hazardous_chemical','composite']
CONDITIONS, ROUTES = ['Good','Poor'], ['Reuse','Recycle','Landfill','Special']
LABELS = {'material': MATERIALS, 'condition': CONDITIONS, 'route': ROUTES, 'threat': ['0','1','2','3']}
MODEL_NAMES = {
    'SC2': ('efficientnet_b0','EfficientNet-B0'),
    'SC3': ('convnext_tiny','ConvNeXt-T'),
    'SC4': ('resnet50','ResNet-50'),
    'SC5': ('mobilenetv3_large_100','MobileNetV3-L'),
    'SC12': ('convnext_small.fb_in22k_ft_in1k','ConvNeXt-S'),
    'SC13': ('vit_small_patch14_dinov2.lvd142m','DINOv2 ViT-S'),
    'SC23': ('vit_base_patch14_dinov2.lvd142m','DINOv2 ViT-B'),
    'SC24': ('vit_base_patch16_siglip_gap_224.webli','SigLIP ViT-B'),
    'SC25': ('vit_small_patch14_reg4_dinov2.lvd142m','DINOv2 ViT-S + registers'),
    'SC7': ('fusion','Fusion (3 backbones)'),
    'SC26': ('fusion','Fusion (4 backbones)'),
    'SC19': ('sc19_convnext_small.fb_in22k_ft_in1k','ConvNeXt-S (alternative schedule)'),
    'SC28': ('singletask_material','Material-only ConvNeXt-S'),
}
SIGNATURES = {
    'SC1':'data_summary.json', 'SC6':'calibration.json', 'SC8':'detection_metrics.json',
    'SC9':'kfold_convnext_small_logits.npz', 'SC10':'extval_trashnet.json',
    'SC10b':'trashnet_finetune.json', 'SC15':'view_aggregation_kfold.json',
    'SC16':'detection_comparison.csv', 'SC18':'end_to_end_results.json',
    'SC20':'kfold_convnext_small5_logits.npz', 'SC21':'gradcam_grid.png',
    'SC22':'pipeline_predictions.csv', 'SC27':'detector_reeval.json',
    'SC29':'tta_results.json', 'SC30':'fig1_labelling_transparent.png',
    'Report_Figures':'report_metrics.json',
    **{sc: f'logits_{name}.npz' for sc,(name,_) in MODEL_NAMES.items()},
}

class EvidenceError(RuntimeError):
    pass

SHA_CACHE = {}

def sha256(path):
    path = Path(path)
    stat = path.stat()
    key = (path.resolve(), stat.st_size, stat.st_mtime_ns)
    if key in SHA_CACHE: return SHA_CACHE[key]
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    digest = h.hexdigest()
    SHA_CACHE[key] = digest
    return digest

def pretty(label):
    return str(label).replace('hazardous_chemical','Hazardous chemical').replace('e_waste','E-waste').replace('_',' ').capitalize()

def _resolve_base():
    if LOCAL_RESULTS_ROOT:
        p = Path(LOCAL_RESULTS_ROOT).expanduser().resolve()
        if not p.is_dir(): raise EvidenceError(f'LOCAL_RESULTS_ROOT is not a directory: {p}')
        return [p]
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent/'Results'/'SC1').is_dir(): return [(parent/'Results').resolve()]
    if Path('/kaggle/input').is_dir(): return [Path('/kaggle/input')]
    raise EvidenceError('Set LOCAL_RESULTS_ROOT or attach completed Kaggle notebook outputs.')

INPUT_BASES = _resolve_base()
OUTPUT = Path(OUTPUT_DIR or ('/kaggle/working/report_figures' if Path('/kaggle/working').is_dir()
                            else INPUT_BASES[0].parent/'Generated_Report_Figures')).resolve()
for b in INPUT_BASES:
    if OUTPUT == b or b in OUTPUT.parents:
        raise EvidenceError('OUTPUT_DIR must be outside input artifacts; inputs are immutable.')
OUTPUT.mkdir(parents=True, exist_ok=True)
# Retain previous managed exports away from the current deliverable. A missing input
# on a later run must never leave an old same-named figure looking newly generated.
history_root = OUTPUT.with_name(OUTPUT.name+'_History')
if history_root.parent.resolve()!=OUTPUT.parent or history_root.resolve()==OUTPUT:
    raise EvidenceError('Invalid output history root')
# Earlier notebook revisions kept history inside the deliverable. Preserve those
# runs outside it so the current folder contains only requested PNG figure files.
legacy_history = OUTPUT/'Previous_Runs'
if legacy_history.is_dir():
    if legacy_history.is_symlink() or legacy_history.resolve().parent!=OUTPUT:
        raise EvidenceError('Invalid legacy history source')
    history_root.mkdir(parents=True,exist_ok=True)
    target = history_root/('Earlier_Runs_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    if target.resolve().parent!=history_root.resolve(): raise EvidenceError('Invalid history target')
    shutil.move(str(legacy_history),str(target))
    print('Preserved earlier output history:',target)
previous_manifest = OUTPUT/'generation_manifest.json'
if previous_manifest.is_file():
    previous = json.loads(previous_manifest.read_text(encoding='utf-8'))
    managed = {Path(p).name for e in previous.get('figures',[]) for p in e.get('files',[])}
    managed.update({'generation_manifest.json','figure_status.csv','README.md',
                    'classification_comparison_verified.csv','view_aggregation_derived.csv',
                    'detector_reevaluation_metrics.json','excluded_report_figures.csv'})
    previous_dir = history_root/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    if previous_dir.resolve().parent!=history_root.resolve(): raise EvidenceError('Invalid archive destination')
    previous_dir.mkdir(parents=True,exist_ok=False)
    for name in sorted(managed):
        source = OUTPUT/name
        if source.parent.resolve()!=OUTPUT: raise EvidenceError('Invalid previous output path')
        if source.is_file(): shutil.move(str(source),str(previous_dir/name))
    print('Archived previous managed figure exports:',previous_dir)
INDEX = defaultdict(list)
for base in INPUT_BASES:
    for folder, dirs, files in os.walk(base):
        dirs[:] = sorted(d for d in dirs if d not in {'Archives','.git','__pycache__'})
        for filename in sorted(files):
            if Path(filename).suffix.lower() in {'.json','.npz','.csv','.png','.yaml','.yml'}:
                INDEX[filename].append(Path(folder)/filename)
SOURCE_EVENTS, MANIFEST = [], []
RESOLVED = {}

def producer_tag(path):
    # Chained outputs can contain other producers. The nearest named producer
    # determines identity; an SC7 ancestor must not make nested SC26 into SC7.
    rx = re.compile(r'(?<![a-z0-9])(SC\d+[a-z]?)(?![a-z0-9])', re.I)
    for part in reversed(Path(path).parts):
        match = rx.search(part)
        if match: return match.group(1).casefold()
    return None

def notebook_mount(path):
    # Ownership is the FIRST input-base/notebooks/owner/slug boundary.
    # Deeper notebooks/... inside all-output are captured copies, not mounts.
    for base in INPUT_BASES:
        try: parts = Path(path).relative_to(base).parts
        except ValueError: continue
        if len(parts)>=3 and parts[0]=='notebooks':
            return base.joinpath(*parts[:3])
    return None

def captured_notebook_path(root, path):
    parts = Path(path).relative_to(root).parts
    return any(part=='notebooks' and len(parts)>=i+3 for i,part in enumerate(parts))

def prefer_direct_notebook_inputs(sc, candidates):
    if not PREFER_DIRECT_NOTEBOOK_INPUTS: return candidates
    preferred = []
    for root in candidates:
        mount = notebook_mount(root)
        if mount is None or captured_notebook_path(mount,root): continue
        matches = (producer_tag(mount.name)==sc.casefold() if sc!='Report_Figures' else
                   bool(re.search(r'(?<![a-z0-9])report[-_ ]figures(?![a-z0-9])',mount.name,re.I)))
        if matches: preferred.append(root)
    if not preferred: return candidates
    ignored = [root for root in candidates if root not in preferred]
    SOURCE_EVENTS.append({'source':sc,'event':'direct_notebook_inputs_preferred',
                          'policy':'PREFER_DIRECT_NOTEBOOK_INPUTS=True',
                          'selected_candidates':list(map(str,preferred)),
                          'ignored_snapshot_candidates':list(map(str,ignored)),
                          'note':'Direct attachment is the configured source scope; this does not establish recency, dataset identity or scientific validity.'})
    print(f'{sc}: using directly attached producer candidates: '+', '.join(map(str,preferred)))
    if ignored: print(f'{sc}: ignored {len(ignored)} captured candidate(s) under the configured direct-input policy.')
    return preferred

def identical_producer_copy(sc, candidates):
    # A matching signature alone is insufficient: another run may have the
    # same PNG/logits but different calibration, metadata or checkpoint files.
    inventories = [{p.relative_to(root).as_posix(): p
                    for p in root.rglob('*') if p.is_file()} for root in candidates]
    reference = inventories[0]
    if any(set(files) != set(reference) for files in inventories[1:]): return None
    for name, first in reference.items():
        paths = [files[name] for files in inventories]
        if any(p.stat().st_size != first.stat().st_size for p in paths[1:]): return None
        if any(sha256(p) != sha256(first) for p in paths[1:]): return None
    # Path ordering is only a representative choice AFTER proving equivalence.
    chosen = min(candidates, key=lambda p: (len(p.parts), str(p)))
    SOURCE_EVENTS.append({'source':sc, 'event':'identical_producer_copies',
                          'selected':str(chosen), 'equivalent_roots':list(map(str,candidates)),
                          'files_compared_per_copy':len(reference),
                          'verification':'same relative file inventory and SHA-256 for every file, including checkpoints'})
    print(f'{sc}: verified {len(candidates)} byte-identical producer copies; using {chosen}')
    return chosen

def resolve_sc(sc):
    if sc in RESOLVED: return RESOLVED[sc]
    if sc in SC_ROOT_OVERRIDES:
        root = Path(SC_ROOT_OVERRIDES[sc]).expanduser().resolve()
        if not root.is_dir(): raise EvidenceError(f'{sc} override not found: {root}')
        RESOLVED[sc] = root
        return root
    direct = [b/sc for b in INPUT_BASES if (b/sc).is_dir()]
    signatures = sorted(set(p.parent for p in INDEX.get(SIGNATURES.get(sc,''), [])))
    # A named mount can be a container (SC30/pipeline_figs), not another run.
    # Use its signature folders when present; keep a direct fallback only for
    # producers whose signature was not exported. Distinct signature folders
    # still undergo full equality checks, even under one named mount.
    direct = [d for d in direct if not any(p==d or d in p.parents for p in signatures)]
    candidates = sorted(set(direct + signatures))
    # SC7 and SC26 share signatures. Discard paths belonging to another named
    # producer before considering generic signatures or identical-copy checks.
    candidates = [p for p in candidates if producer_tag(p) in (None,sc.casefold())]
    tagged = [p for p in candidates if producer_tag(p)==sc.casefold()]
    if tagged: candidates = tagged
    candidates = prefer_direct_notebook_inputs(sc,candidates)
    if sc == 'SC7':
        if not tagged:
            raise EvidenceError('SC7 fusion identity needs an SC7-tagged path or explicit SC_ROOT_OVERRIDES; a unique generic fusion filename is insufficient.')
        candidates = [p for p in candidates if not (p/'fusion_backbones_used.json').exists()]
    if sc == 'SC26':
        explicit = [p for p in candidates if (p/'fusion_backbones_used.json').exists()]
        if explicit: candidates = explicit
        elif not tagged:
            raise EvidenceError('SC26 fusion identity needs an SC26-tagged path, its fusion_backbones_used.json, or explicit SC_ROOT_OVERRIDES.')
    if len(candidates) > 1:
        chosen = identical_producer_copy(sc,candidates)
        if chosen is None:
            raise EvidenceError(f'Ambiguous {sc}: producer copies differ: {list(map(str,candidates))}. Set SC_ROOT_OVERRIDES[{sc!r}] to the intended producer/version directory.')
        candidates = [chosen]
    if len(candidates) == 0:
        raise FileNotFoundError(f'No {sc} artifacts; attach {SIGNATURES.get(sc,sc)} or set its override.')
    RESOLVED[sc] = candidates[0]
    return candidates[0]

def artifact(sc, name, required=True):
    try:
        root = resolve_sc(sc)
        # A companion at the selected producer root belongs to that producer;
        # do not replace it with a same-named file from captured upstream inputs.
        direct = root/name
        if direct.is_file() and not captured_notebook_path(root,direct.parent): return direct
        paths = sorted(root.rglob(name)) if '/' not in name and '\\' not in name else [direct]
        paths = [p for p in paths if p.is_file()]
        paths = [p for p in paths if not captured_notebook_path(root,p.parent)]
        paths = [p for p in paths if producer_tag(p.parent.relative_to(root)) in (None,sc.casefold())]
        if len(paths) > 1:
            if len({(p.stat().st_size,sha256(p)) for p in paths}) != 1:
                raise EvidenceError(f'Ambiguous {sc}/{name}: copies differ: {paths}; use a narrower SC_ROOT_OVERRIDES[{sc!r}].')
            chosen = min(paths, key=lambda p: (len(p.parts), str(p)))
            SOURCE_EVENTS.append({'source':sc,'event':'identical_artifact_copies','artifact':name,
                                  'selected':str(chosen),'equivalent_files':list(map(str,paths)),
                                  'sha256':sha256(chosen)})
            paths = [chosen]
        if not paths: raise FileNotFoundError(f'Missing {sc}/{name}')
        return paths[0]
    except FileNotFoundError:
        if required: raise
        return None

def read_json(sc, name):
    p = artifact(sc,name)
    with p.open(encoding='utf-8-sig') as f: return json.load(f), p

def register(name, status, sources=(), evaluation=None, notes='', files=(), **extra):
    entry = {'figure':name,'status':status,'evaluation':evaluation,'notes':notes,
             'sources':[{'path':str(p),'sha256':sha256(p)} for p in sources if p is not None],
             'files':[str(p) for p in files], **extra}
    MANIFEST.append(entry)
    return entry

def save(fig, name, sources=(), evaluation=None, notes='', status='generated', data=None):
    # Plot sections own their layout so compact report panels retain legible type.
    # Evaluation metadata belongs in the manifest and report caption; avoid a
    # tiny boilerplate footer changing every reference image's aspect ratio.
    # Keep experiment identifiers in provenance files, never in plotted text.
    from matplotlib.text import Text
    fig.canvas.draw()
    visible_labels = [artist.get_text() for artist in fig.findobj(Text)
                      if artist.get_visible() and artist.get_text()]
    forbidden = [label for label in visible_labels
                 if re.search(r'\bSC\d+[a-z]?\b|\bnotebooks?\b|\.ipynb\b', label, re.I)]
    if forbidden:
        plt.close(fig)
        raise EvidenceError(f'{name} contains an experiment identifier in a figure label: {forbidden}')
    p = OUTPUT/(Path(name).stem+'.png')
    fig.savefig(p, dpi=600, bbox_inches='tight', facecolor='white')
    outputs = [p]
    if SHOW_FIGURES:
        from IPython.display import display
        display(fig)
    plt.close(fig)
    if data is not None:
        p = OUTPUT/(Path(name).stem+'_data.json')
        p.write_text(json.dumps(data,indent=2,default=lambda x:x.tolist() if isinstance(x,np.ndarray) else float(x)),encoding='utf-8')
        outputs.append(p)
    register(name,status,sources,evaluation,notes,outputs)

def unavailable(names, reason, instruction, sources=()):
    for name in names:
        register(name,'missing',sources,notes=reason,action=instruction)

def run_section(label, fn, figures):
    try:
        fn()
        print(f'{label}: complete')
    except (FileNotFoundError,EvidenceError,ValueError,KeyError,AssertionError,ImportError) as exc:
        print(f'{label}: blocked: {exc}')
        existing = {e['figure'] for e in MANIFEST}
        unavailable([n for n in figures if n not in existing],str(exc),
                    'Attach the named producer outputs, resolve any duplicate mount using SC_ROOT_OVERRIDES, then rerun.')

def style_axes(ax, title='', xlabel=None, ylabel=None):
    ax.set_title(title,loc='left',pad=14)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(axis='x',alpha=.15)
    ax.set_axisbelow(True)

print('Input roots:', ', '.join(map(str,INPUT_BASES)))
print('Output directory:', OUTPUT)

## Audited numerical primitives and cohort verification

In [ ]:
def sigmoid(z):
    return (1/(1+np.exp(-np.clip(np.asarray(z),-80,80)))).astype(np.float32)

def softmax(z):
    z=np.asarray(z,dtype=np.float32); z=z-z.max(axis=-1,keepdims=True)
    e=np.exp(z); return e/e.sum(axis=-1,keepdims=True)

def corn_cumulative(z): return np.cumprod(sigmoid(z),axis=-1)

def cumulative_to_probs(c):
    c=np.asarray(c)
    return np.maximum(np.concatenate([1-c[...,:1],c[...,:-1]-c[...,1:],c[...,-1:]],axis=-1),1e-6)

def corn_probs(z): return cumulative_to_probs(corn_cumulative(z))

def probabilities(logits):
    return {'material':softmax(logits['material']), 'condition':sigmoid(logits['condition']),
            'route':softmax(logits['route']), 'threat':corn_probs(logits['threat'])}

def predictions(logits):
    p=probabilities(logits)
    return {'material':p['material'].argmax(1),'condition':(p['condition']>.5).astype(int),
            'route':p['route'].argmax(1),'threat':(corn_cumulative(logits['threat'])>.5).sum(1)}

def ece(p,y,bins=15):
    p=np.asarray(p); y=np.asarray(y,dtype=int)
    conf=p.max(1); correct=(p.argmax(1)==y).astype(float); edges=np.linspace(0,1,bins+1)
    return float(sum(abs(correct[m].mean()-conf[m].mean())*m.sum()/len(y)
                     for lo,hi in zip(edges[:-1],edges[1:]) if (m:=(conf>lo)&(conf<=hi)).sum()))

def itemize_file(path,split='test'):
    with np.load(path,allow_pickle=False) as d:
        items=d[f'{split}_items'].astype(str)
        logits={h:d[f'{split}_log_{h}'] for h in HEADS}
        labels={h:d[f'{split}_y_{h}'].astype(int) for h in HEADS}
    unique=list(dict.fromkeys(items.tolist()))
    groups={it:np.flatnonzero(items==it) for it in unique}
    if not unique: raise EvidenceError(f'No {split} items in {path}')
    for h in HEADS:
        if len(logits[h])!=len(items) or not np.isfinite(logits[h]).all():
            raise EvidenceError(f'Invalid {h} logits in {path}')
        if any(len(np.unique(labels[h][ix])) != 1 for ix in groups.values()):
            raise EvidenceError(f'Conflicting per-view {h} labels within an item in {path}')
    agg={h:np.stack([logits[h][ix].astype(np.float64).mean(0) for ix in groups.values()]).astype(np.float32) for h in HEADS}
    ys={h:np.asarray([labels[h][ix[0]] for ix in groups.values()]) for h in HEADS}
    return {'items':np.asarray(unique),'logits':agg,'labels':ys,'view_logits':logits,
            'groups':groups,'path':Path(path),'n_views':len(items),'split':split}

def model_data(sc,split='test'):
    model,_=MODEL_NAMES[sc]
    return itemize_file(artifact(sc,f'logits_{model}.npz'),split)

def assert_same_cohort(a,b):
    if set(a['items'])!=set(b['items']):
        raise EvidenceError(f'Cohorts differ: {a["path"]} ({len(a["items"])} items) versus {b["path"]} ({len(b["items"])}).')
    ix={v:i for i,v in enumerate(b['items'])}; order=[ix[v] for v in a['items']]
    if any(not np.array_equal(a['labels'][h],b['labels'][h][order]) for h in HEADS):
        raise EvidenceError(f'Ground-truth labels differ for {a["path"]} and {b["path"]}')

def scalars(d):
    y=d['labels']; p=probabilities(d['logits']); q=predictions(d['logits'])
    return {'material_f1':f1_score(y['material'],q['material'],labels=range(10),average='macro',zero_division=0),
            'material_acc':accuracy_score(y['material'],q['material']),
            'route_f1':f1_score(y['route'],q['route'],labels=range(4),average='macro',zero_division=0),
            'condition_auroc':roc_auc_score(y['condition'],p['condition']),
            'threat_qwk':cohen_kappa_score(y['threat'],q['threat'],labels=range(4),weights='quadratic')}

def verified_metric(sc,d):
    model,_=MODEL_NAMES[sc]; rep,path=read_json(sc,f'metrics_{model}.json')
    actual=scalars(d); saved={'material_f1':rep['scalars']['material']['macro_f1'],
                             'material_acc':rep['scalars']['material']['acc'],
                             'route_f1':rep['scalars']['route']['macro_f1'],
                             'condition_auroc':rep['scalars']['condition']['auroc'],
                             'threat_qwk':rep['scalars']['threat']['qwk']}
    if rep['n_items']!=len(d['items']): raise EvidenceError(f'{sc} metrics/logit item-count mismatch')
    if any(not np.isclose(actual[k],saved[k],atol=2e-6,rtol=0) for k in saved):
        raise EvidenceError(f'{sc} saved metrics disagree with actual logits: computed={actual}, saved={saved}')
    return actual,path

CV_CACHE = {}
def cv_data(source=None):
    requested=source or CV_SOURCE
    if requested in CV_CACHE: return CV_CACHE[requested]
    choices=['SC20','SC9'] if requested=='auto' else [requested]
    if any(sc not in ['SC9','SC20'] for sc in choices): raise EvidenceError('CV_SOURCE must be auto, SC9, or SC20')
    for sc in choices:
        tag='convnext_small5' if sc=='SC20' else 'convnext_small'
        try:
            d=itemize_file(artifact(sc,f'kfold_{tag}_logits.npz'))
            fold_verified=False; fold_paths=[]
            if sc=='SC20':
                summary,_=read_json('SC1','data_summary.json')
                if len(d['items'])!=sum(summary['items'].values()):
                    raise EvidenceError('SC20 pooled item count differs from the currently attached SC1 release.')
                test=model_data('SC12'); lookup={it:i for i,it in enumerate(d['items'])}
                if not set(test['items']).issubset(lookup): raise EvidenceError('SC20 does not cover the current SC12 test items.')
                order=[lookup[it] for it in test['items']]
                if any(not np.array_equal(d['labels'][h][order],test['labels'][h]) for h in HEADS):
                    raise EvidenceError('SC20/current SC12 ground-truth label mismatch.')
                fold_paths=[artifact('SC20',f'kfold5_fold{k}_logits.npz',required=False) for k in range(5)]
                if all(p is not None for p in fold_paths):
                    seen=set(); concatenated=defaultdict(list)
                    for k,path in enumerate(fold_paths):
                        with np.load(path,allow_pickle=False) as z:
                            if 'fold' not in z or np.asarray(z['fold']).tolist()!=[k]: raise EvidenceError(f'SC20 fold ID mismatch in {path}')
                            item_set=set(z['test_items'].astype(str))
                            if seen & item_set: raise EvidenceError('SC20 items appear in multiple held-out folds.')
                            seen.update(item_set)
                            for key in z.files:
                                if key.startswith('test_'): concatenated[key].append(z[key])
                    if seen!=set(d['items']): raise EvidenceError('SC20 fold exports and pooled archive cover different items.')
                    with np.load(d['path'],allow_pickle=False) as merged:
                        for key,arrays in concatenated.items():
                            if not np.array_equal(np.concatenate(arrays),merged[key]): raise EvidenceError(f'SC20 merged values do not reproduce from five fold files: {key}')
                    fold_verified=True
                elif requested=='auto':
                    raise EvidenceError('SC20 merge found but all five per-fold exports are needed for automatic preference; set CV_SOURCE=SC20 for explicitly unverified fold count.')
            current_summary,_=read_json('SC1','data_summary.json')
            is_historical=len(d['items'])!=sum(current_summary['items'].values())
            d.update({'producer':sc,'tag':tag,'fold_count_verified':fold_verified,
                      'fold_paths':[p for p in fold_paths if p is not None],
                      'label':('SC20 five disjoint held-out folds verified' if fold_verified else
                               f'{sc} '+('historical ' if is_historical else 'pooled ')+'OOF (fold count unverified)' if sc=='SC9' else 'SC20 pooled OOF (fold count unverified)')})
            CV_CACHE[requested]=d
            print('Selected OOF evaluation:',d['label'],f'| {len(d["items"])} items')
            return d
        except (FileNotFoundError,EvidenceError) as exc:
            if requested!='auto' or sc=='SC9': raise
            SOURCE_EVENTS.append({'source':sc,'event':'not_selected','reason':str(exc)})
            print('SC20 not selected:',exc)
    raise EvidenceError('No eligible OOF artifacts were attached.')

def diagnostic_data(choice):
    return cv_data(None if choice=='cv' else choice) if choice in ['cv','SC9','SC20'] else model_data(choice)

## Dataset distributions: item counts, never augmented image counts

In [ ]:
def dataset_figures():
    summary,sp=read_json('SC1','data_summary.json'); config,cp=read_json('SC1','config.json')
    if config['MATERIALS']!=MATERIALS or config['ROUTES']!=ROUTES: raise EvidenceError('Unexpected label encoding in SC1.')
    total=sum(summary['items'].values())
    for key in ['material_counts','route_counts','threat_counts']:
        if sum(summary[key].values())!=total: raise EvidenceError(f'{key} does not sum to {total} items.')
    # Preserve the original half-textwidth, count-only distribution design.
    short={'hazardous_chemical':'haz. chem.','e_waste':'e-waste'}
    ordered=sorted(MATERIALS,key=lambda k:-summary['material_counts'].get(k,0))
    vals=[summary['material_counts'].get(k,0) for k in ordered]
    fig=plt.figure(figsize=(2.797,2.797*859/1759)); ax=fig.add_axes([.235,.015,.755,.905])
    bars=ax.barh(np.arange(10),vals,color=GREEN,height=.62)
    ax.set_yticks(np.arange(10),[short.get(k,k) for k in ordered],fontsize=8)
    ax.invert_yaxis(); ax.bar_label(bars,padding=3,fontsize=7.2,fontweight='bold')
    ax.set_xlim(0,max(vals)*1.15); ax.set_xticks([]); ax.tick_params(axis='y',length=0,pad=3)
    for spine in ax.spines.values(): spine.set_visible(False)
    fig.text(.98,.985,f'total = {total}',ha='right',va='top',fontsize=7.2,fontstyle='italic')
    save(fig,'material_distribution.png',[sp,cp],f'SC1 | {total} physical items | all three splits',
         'Counts are item-level; training images include online/offline augmentation and are not independent items.',data=summary['material_counts'])
    fig=plt.figure(figsize=(2.797,2.797*734/1759))
    axes=[fig.add_axes([.155,.02,.300,.840]),fig.add_axes([.640,.02,.300,.840])]
    threat_keys=['0','1','2','3'] if summary['threat_counts'].get('0',0) else ['1','2','3']
    for ax,keys,counts,title,colors in [(axes[0],ROUTES,summary['route_counts'],'Route',ROUTE_COLORS),
            (axes[1],threat_keys,summary['threat_counts'],'Threat',[GRAY,GREEN,ORANGE,RED] if '0' in threat_keys else [GREEN,ORANGE,RED])]:
        v=[counts.get(k,0) for k in keys]
        labels=[k.lower() for k in keys] if title=='Route' else [f'level {k}' for k in keys]
        b=ax.barh(np.arange(len(keys)),v,color=colors,height=.56)
        ax.set_yticks(np.arange(len(keys)),labels,fontsize=7.5); ax.invert_yaxis()
        ax.bar_label(b,padding=2,fontsize=7.2,fontweight='bold'); ax.set_xlim(0,max(v)*1.30)
        ax.set_xticks([]); ax.tick_params(axis='y',length=0,pad=2); ax.set_title(title,fontsize=8.5,pad=5)
        for spine in ax.spines.values(): spine.set_visible(False)
    save(fig,'route_threat_distribution.png',[sp,cp],f'SC1 | {total} physical items | all three splits',data={k:summary[k] for k in ['route_counts','threat_counts']})

run_section('Dataset distributions',dataset_figures,['material_distribution.png','route_threat_distribution.png'])

## Backbone comparisons: independently resolve each producer and compare identical items

In [ ]:
BENCHMARK_SC = ['SC2','SC3','SC4','SC5','SC12','SC13','SC23','SC24','SC25','SC26']
if INCLUDE_BASELINE_FUSION: BENCHMARK_SC.append('SC7')
def comparison_figures():
    reference=model_data('SC12'); rows=[]; sources=[]; omissions=[]
    for sc in BENCHMARK_SC:
        try:
            d=model_data(sc); assert_same_cohort(reference,d); row,mp=verified_metric(sc,d)
            rows.append({'source':sc,'model':MODEL_NAMES[sc][1],**row}); sources.extend([d['path'],mp])
        except FileNotFoundError as exc:
            omissions.append(f'{sc}: {exc}')
    df=pd.DataFrame(rows)
    if len(df)<2: raise EvidenceError('At least two verified classifiers on identical items are needed.')
    df.to_csv(OUTPUT/'classification_comparison_verified.csv',index=False)
    mapping={'material_f1':('Material macro-F1','cls_compare_material_f1.png'),
             'threat_qwk':('Threat quadratic weighted kappa','cls_compare_threat_qwk.png'),
             'route_f1':('Route macro-F1','cls_compare_route_f1.png'),
             'condition_auroc':('Condition AUROC','cls_compare_condition_auroc.png')}
    for key,(title,name) in mapping.items():
        ranked=df.sort_values([key,'source'],ascending=[False,True]); fig,ax=plt.subplots(figsize=(6.5,4.5))
        c=[ORANGE if sc=='SC12' else GRAY if sc in ['SC7','SC26'] else TEAL if sc in ['SC13','SC23','SC24','SC25'] else BLUE for sc in ranked.source]
        names={'SC12':'ConvNeXt-S','SC26':'Fusion (4 backbones)','SC7':'Fusion (3 backbones)','SC25':'DINOv2 ViT-S + reg.'}
        bars=ax.barh([names.get(sc,MODEL_NAMES[sc][1]) for sc in ranked.source],ranked[key],color=c,height=.66)
        ax.invert_yaxis(); ax.bar_label(bars,fmt='%.3f',padding=4,fontsize=9)
        ax.set_xlim(min(.5,float(ranked[key].min())-.05),max(.95,float(ranked[key].max())+.085))
        ax.set_xlabel(title); ax.grid(axis='x',alpha=.14); ax.set_axisbelow(True)
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color=BLUE,label='CNN'),Patch(color=ORANGE,label='Reference'),Patch(color=TEAL,label='Transformer'),Patch(color=GRAY,label='Ensemble')],
                  loc='upper center',bbox_to_anchor=(.5,-.17),ncol=4,frameon=False,fontsize=8.5)
        fig.subplots_adjust(left=.32,right=.97,top=.97,bottom=.22)
        save(fig,name,sources,f'Held-out common cohort | {len(reference["items"])} items | uncalibrated logits',
             'Exploratory benchmark; no post-test model adoption. SC19 uses a different release and is excluded. SC28 untrained heads are excluded. '+ '; '.join(omissions),data=ranked.to_dict('records'))

run_section('Verified classifier comparisons',comparison_figures,[f'cls_compare_{k}.png' for k in ['material_f1','threat_qwk','route_f1','condition_auroc']])

## Historical cross-validation versus current held-out scores

In [ ]:
def cv_figure():
    cv=cv_data(); ci,cp=read_json(cv['producer'],f'kfold_{cv["tag"]}_ci.json'); test=model_data(PRIMARY_SC)
    tm,_=verified_metric(PRIMARY_SC,test); cvm=scalars(cv)
    if ci['n_items']!=len(cv['items']): raise EvidenceError('Selected CI count does not match OOF logits.')
    for key,v in ci['ci'].items():
        if not np.isclose(cvm[key],v['point'],atol=5.1e-5,rtol=0): raise EvidenceError(f'Selected CI point mismatch: {key}')
    keys=['material_acc','material_f1','threat_qwk','condition_auroc','route_f1']
    names=['material accuracy','material macro-F1','threat QWK','condition AUROC','route macro-F1']
    fig,ax=plt.subplots(figsize=(7.2,3.9)); yy=np.arange(len(keys))
    point=np.array([ci['ci'][k]['point'] for k in keys]); lo=np.array([ci['ci'][k]['lo'] for k in keys]); hi=np.array([ci['ci'][k]['hi'] for k in keys])
    fold_label='Five-fold out-of-fold' if cv['fold_count_verified'] else 'Pooled out-of-fold'
    ax.errorbar(point,yy-.13,xerr=np.stack([point-lo,hi-point]),fmt='o',capsize=4,color=BLUE,
                label=f'{fold_label}, n={len(cv["items"])}',ms=5,lw=1.5)
    ax.scatter([tm[k] for k in keys],yy+.13,marker='D',color=ORANGE,s=35,label=f'{MODEL_NAMES[PRIMARY_SC][1]} test, n={len(test["items"])}',zorder=4)
    for j,k in enumerate(keys):
        ax.annotate(f'{point[j]:.3f}',(point[j],j-.13),xytext=(0,8),textcoords='offset points',ha='center',va='bottom',fontsize=8,color=BLUE)
        ax.annotate(f'{tm[k]:.3f}',(tm[k],j+.13),xytext=(6,0),textcoords='offset points',va='center',fontsize=8,color=ORANGE)
    ax.set_yticks(yy,names); ax.invert_yaxis()
    ax.set_xlim(min(.55,float(lo.min())-.035,min(tm.values())-.035),max(.96,float(hi.max())+.025,max(tm.values())+.065))
    ax.set_ylim(len(keys)-.45,-.80); ax.set_xlabel('Score (OOF: supplied 95% bootstrap confidence interval)')
    ax.grid(axis='x',alpha=.16); ax.set_axisbelow(True)
    ax.legend(loc='upper left',bbox_to_anchor=(0,1.19),fontsize=8,frameon=False)
    fig.subplots_adjust(left=.235,right=.975,bottom=.16,top=.82)
    ev=f'{cv["label"]} | n={len(cv["items"])} | {PRIMARY_SC}: separate held-out evaluation'
    save(fig,'kfold_ci.png',[cp,cv['path'],test['path'],*cv['fold_paths']],ev,
         'OOF and held-out scores are separate evaluations, not paired model comparisons. The supplied starting SC9 export covers 457 historical items and lacks executed fold metadata; any rerun is labeled by its actual item count. SC20 is preferred automatically only when all five disjoint fold exports exactly reproduce its pooled archive, its count matches SC1, and current SC12 test labels agree. Full dataset identity and train/validation separation still require upstream manifests/run fingerprints.',data={'producer':cv['producer'],'fold_count_verified':cv['fold_count_verified'],'ci':ci,'held_out':tm})

run_section('Historical CV intervals',cv_figure,['kfold_ci.png'])

## Confusion matrices and ROC curves from actual per-view logits

In [ ]:
def confusion_figures():
    d=diagnostic_data(DIAGNOSTIC_SC)
    q=predictions(d['logits']); ev=f'{d.get("label",DIAGNOSTIC_SC+" held-out test")} | {len(d["items"])} items'
    for head in HEADS:
        labs=LABELS[head]; cm=confusion_matrix(d['labels'][head],q[head],labels=range(len(labs)))
        support=cm.sum(1); norm=np.divide(cm,support[:,None],out=np.zeros_like(cm,dtype=float),where=support[:,None]>0)
        shown=list(range(len(labs)))
        if head=='threat' and cm[0].sum()==0 and cm[:,0].sum()==0: shown=[1,2,3]
        display_cm=cm[np.ix_(shown,shown)]; display_norm=norm[np.ix_(shown,shown)]
        size=(6.1,5.5) if head=='material' else (4.6,4.0)
        fig,ax=plt.subplots(figsize=size); im=ax.imshow(display_norm,cmap='Blues',vmin=0,vmax=1)
        short={'hazardous_chemical':'haz. chem.','e_waste':'e-waste'}
        ticks=[('level '+labs[k]) if head=='threat' else short.get(labs[k],labs[k].lower()) for k in shown]
        ax.set_xticks(range(len(shown)),ticks,rotation=45 if head=='material' else 0,ha='right' if head=='material' else 'center')
        ax.set_yticks(range(len(shown)),ticks); ax.tick_params(length=0,labelsize=9)
        for i in range(len(shown)):
            for j in range(len(shown)):
                if display_cm[i,j]: ax.text(j,i,str(display_cm[i,j]),ha='center',va='center',fontsize=10 if head=='material' else 13,color='white' if display_norm[i,j]>.53 else INK)
        ax.set_xlabel('predicted'); ax.set_ylabel('true')
        colorbar=fig.colorbar(im,ax=ax,fraction=.045,pad=.035); colorbar.set_label('row fraction',fontsize=9)
        fig.subplots_adjust(left=.19 if head=='material' else .17,right=.89,bottom=.24 if head=='material' else .16,top=.97)
        save(fig,f'confusion_{head}.png',[d['path']],ev,
             'Color encodes row fraction; nonzero cells show raw item counts. Threat decoder = count of cumulative CORN probabilities > 0.5, matching full_report. Reserved threat level 0 is omitted visually only when both its true row and predicted column are empty; full four-level arrays are retained here.',status='derived',data={'labels':labs,'counts':cm,'row_fraction':norm,'displayed_labels':[labs[k] for k in shown],'support':support})

def roc_figures():
    d=diagnostic_data(DIAGNOSTIC_SC); p=probabilities(d['logits'])
    ev=f'{d.get("label",DIAGNOSTIC_SC+" held-out test")} | {len(d["items"])} item-level mean-logit predictions'
    for head in ['condition','route']:
        fig,ax=plt.subplots(figsize=(4.9,4.2)); curves={}
        for k,lab in ([(1,'Poor')] if head=='condition' else enumerate(ROUTES)):
            y=(d['labels'][head]==k).astype(int); score=p[head] if head=='condition' else p[head][:,k]
            if len(np.unique(y))<2: raise EvidenceError(f'ROC undefined: no positive or negative examples for {head}/{lab}')
            fpr,tpr,threshold=roc_curve(y,score); auc=roc_auc_score(y,score)
            ax.step(fpr,tpr,where='post',linewidth=2.0,color=BLUE if head=='condition' else ROUTE_COLORS[k],label=f'{lab.lower()} (AUROC {auc:.3f})')
            curves[str(lab)]={'fpr':fpr,'tpr':tpr,'auroc':auc,'positive_support':int(y.sum())}
        ax.plot([0,1],[0,1],color=GRAY,ls='--',lw=1); ax.set(xlim=(0,1),ylim=(0,1.02),xlabel='False-positive rate',ylabel='True-positive rate')
        ax.grid(alpha=.15); ax.legend(loc='lower right',fontsize=9,frameon=True,facecolor='white',edgecolor='#D4D8DC')
        fig.subplots_adjust(left=.15,right=.97,bottom=.15,top=.97)
        save(fig,f'{head}_roc.png',[d['path']],ev,'Empirical step curves; no interpolation into fictitious observations, no synthetic data.',status='derived',data=curves)

run_section('Actual confusion matrices',confusion_figures,[f'confusion_{h}.png' for h in HEADS])
run_section('Actual ROC curves',roc_figures,['condition_roc.png','route_roc.png'])

## Calibration: establish which model owns the temperatures before applying them

In [ ]:
def calibration_figure():
    cal,cp=read_json('SC6','calibration.json'); owners=[]; candidates={}
    for sc in list(dict.fromkeys(BENCHMARK_SC+['SC7'])):
        try:
            d=model_data(sc); p=probabilities(d['logits']); vals={h:ece(np.stack([1-p[h],p[h]],1) if h=='condition' else p[h],d['labels'][h]) for h in HEADS}
            if all(np.isclose(vals[h],cal[h]['ece_before'],atol=3e-6,rtol=0) for h in HEADS): owners.append(sc); candidates[sc]=d
        except FileNotFoundError: pass
    if len(owners)!=1: raise EvidenceError(f'Calibration model identity cannot be verified uniquely (owners={owners}). Re-export model/run identity with temperatures.')
    sc=owners[0]; d=candidates[sc]; raw=probabilities(d['logits'])
    scaled={h:d['logits'][h]/np.asarray(cal[h]['T'],dtype=np.float32) for h in HEADS}; post=probabilities(scaled)
    before={h:ece(np.stack([1-raw[h],raw[h]],1) if h=='condition' else raw[h],d['labels'][h]) for h in HEADS}
    after={h:ece(np.stack([1-post[h],post[h]],1) if h=='condition' else post[h],d['labels'][h]) for h in HEADS}
    if any(not np.isclose(after[h],cal[h]['ece_after'],atol=3e-6,rtol=0) for h in HEADS): raise EvidenceError('Saved calibrated ECE does not reproduce; do not transplant temperatures.')
    # The report's two panels address different questions and different cohorts.
    # Left: the selected OOF material reliability. Right: the verified owner of SC6.
    oof=cv_data(); p=probabilities(oof['logits'])['material']
    conf=p.max(1); correct=(p.argmax(1)==oof['labels']['material']); xs=[]; ys=[]; sizes=[]
    edges=np.linspace(0,1,16); frequencies=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(conf>lo)&(conf<=hi); frequencies.append(float(mask.sum()/len(conf)))
        if mask.any(): xs.append(float(conf[mask].mean())); ys.append(float(correct[mask].mean())); sizes.append(int(mask.sum()))
    oof_ece=ece(p,oof['labels']['material'])
    reliability={'mean_confidence':xs,'accuracy':ys,'counts':sizes,'bin_edges':edges,
                 'bin_fractions':frequencies,'ece':oof_ece,'producer':oof['producer'],'n_items':len(conf)}
    fig,axes=plt.subplots(1,2,figsize=(10.3,4.35)); ax=axes[0]
    ax.bar((edges[:-1]+edges[1:])/2,frequencies,width=1/15*.90,color='#DEE3E7',label='fraction of items per bin',zorder=1)
    ax.plot([0,1],[0,1],'--',color=GRAY,lw=1.2,label='perfect calibration')
    ax.plot(xs,ys,'o-',lw=1.8,ms=4.5,label=f'accuracy (ECE {oof_ece:.3f})',color=BLUE,zorder=3)
    ax.set(xlim=(0,1),ylim=(0,1.03),xlabel='mean predicted confidence',ylabel='accuracy / fraction of items')
    ax.set_title(f'Material reliability: out-of-fold (n={len(conf)})',pad=10)
    ax.grid(alpha=.14); ax.legend(loc='upper left',fontsize=8,frameon=False)
    ax=axes[1]; order=['condition','route','material','threat']; x=np.arange(4)
    for offset,values,label,color in [(-.18,before,'before',GRAY),(.18,after,'after',GREEN)]:
        bars=ax.bar(x+offset,[values[h] for h in order],width=.34,color=color,label=label,zorder=3)
        ax.bar_label(bars,fmt='%.3f',padding=4,fontsize=8)
    ax.set_xticks(x,order); ax.set_ylim(0,max(*before.values(),*after.values())*1.32)
    ax.set_ylabel('expected calibration error (ECE)'); ax.set_title(f'{MODEL_NAMES[sc][1]} calibration (n={len(d["items"])})',pad=10)
    ax.grid(axis='y',alpha=.14); ax.set_axisbelow(True); ax.legend(fontsize=8,frameon=False)
    fig.subplots_adjust(left=.075,right=.98,top=.88,bottom=.19,wspace=.29)
    fig.text(.5,.025,'Separate evaluations: OOF reliability at left; validation-fitted temperatures evaluated on the verified owner at right.',ha='center',fontsize=8,color='#58636B')
    save(fig,'calibration_reliability.png',[cp,d['path'],oof['path'],*oof['fold_paths']],f'Left: {oof["label"]}, n={len(conf)} | right: verified {sc}, n={len(d["items"])} test items',
         'Left uses uncalibrated mean-logit OOF material predictions; SC6 temperatures are never applied to this OOF archive. Right owner inferred by matching all four before-ECE values and verified by reproducing all four after-ECE values. These supplied temperatures do not belong to SC12 or SC9. Threat temperature is a supplied 3-vector; current source notebook differs. Threat ECE uses class argmax, whereas threat QWK uses cumulative threshold decoding.',data={'owner':sc,'before':before,'after':after,'temperatures':cal,'reliability':reliability,'oof_owner':oof['producer']})

run_section('Calibration provenance and plot',calibration_figure,['calibration_reliability.png'])

## Training traces: logged epochs only

In [ ]:
def training_figure():
    sc=PRIMARY_SC; name,_=MODEL_NAMES[sc]; p=artifact(sc,f'history_{name}.csv'); h=pd.read_csv(p)
    required=['epoch','train_loss','val_loss','val_material_f1','val_route_f1','val_threat_qwk','val_condition_f1']
    if any(k not in h for k in required): raise EvidenceError(f'History is missing required columns: {p}')
    fig,axes=plt.subplots(1,2,figsize=(10.3,4.0)); x=h.epoch
    score=.4*h.val_route_f1+.4*h.val_threat_qwk+.2*h.val_material_f1
    selected_epoch=float(x.iloc[int(np.argmax(score.to_numpy()))])
    axes[0].plot(x,h.train_loss,marker='o',ms=3,lw=2,color=BLUE)
    axes[0].set(xlabel='epoch (recorded index)',ylabel='training loss'); axes[0].grid(alpha=.15)
    for key,label,color in [('val_material_f1','material macro-F1',BLUE),('val_route_f1','route macro-F1',TEAL),('val_threat_qwk','threat QWK',ORANGE),('val_condition_f1','condition F1',RED)]: axes[1].plot(x,h[key],marker='o',ms=3,lw=1.8,label=label,color=color)
    axes[1].axvline(selected_epoch,color=GRAY,lw=1.2,ls='--',alpha=.8)
    axes[1].set(xlabel='epoch (recorded index)',ylabel='validation score',ylim=(min(-.05,float(h.val_threat_qwk.min())-.05),1.03))
    axes[1].legend(fontsize=8,loc='lower right',ncol=2,frameon=True,facecolor='white',edgecolor='#D4D8DC'); axes[1].grid(alpha=.15)
    axes[1].set_title(f'Checkpoint selected at epoch {selected_epoch:g}',fontsize=10,pad=8)
    fig.subplots_adjust(left=.07,right=.985,bottom=.16,top=.90,wspace=.23)
    save(fig,'convnext_s_training_curves.png',[p],f'{sc} | {len(h)} recorded epochs | validation evaluation, not test curves',
         'No synthesized epochs. Dashed line: first maximum of source selection blend 0.4 route F1 + 0.4 threat QWK + 0.2 material F1. Training/validation losses differ in weighting/smoothing, EMA and evaluation/TTA; their gap is not an isolated overfitting measure. Filename retained for current report; change caption if PRIMARY_SC changes.',data={'selected_epoch':selected_epoch,'history':h.to_dict('records')})

run_section('Logged training history',training_figure,['convnext_s_training_curves.png'])

## End-to-end and external validation: incompatible denominators stay visible

In [ ]:
def e2e_figure():
    rep,rp=read_json('SC18','end_to_end_results.json'); d=model_data('SC12'); actual,_=verified_metric('SC12',d)
    for key,value in rep['oracle'].items():
        if key in actual and key!='condition_auroc' and not np.isclose(value,actual[key],atol=2e-6): raise EvidenceError(f'SC18 oracle {key} does not match SC12')
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    values=[rep['oracle']['material_acc'],rep['end_to_end']['material_acc']]
    bars=ax.bar(['oracle crop\n(ground-truth box)','end-to-end\n(detector crop)'],values,color=[BLUE,ORANGE],width=.56)
    ax.bar_label(bars,fmt='%.3f',padding=5,fontsize=10)
    ax.text(1,values[1]*.52,f'detection rate\n{rep["detect_rate"]:.1%}',ha='center',va='center',color='white',fontsize=10)
    delta=values[1]-values[0]
    label='material accuracy unchanged' if np.isclose(delta,0,atol=1e-12) else f'material accuracy change: {delta:+.3f}'
    ax.text(.5,1.045,label,ha='center',va='bottom',fontsize=10)
    ax.set_ylim(0,1.12); ax.set_ylabel('material accuracy (test items)'); ax.grid(axis='y',alpha=.15); ax.set_axisbelow(True)
    fig.subplots_adjust(left=.14,right=.97,bottom=.18,top=.94)
    save(fig,'end_to_end.png',[rp,d['path']],f'SC18 | {len(d["items"])} test items | detection rate {rep["detect_rate"]:.0%}',
         'The material scores tie, but route and condition change; do not claim globally lossless end-to-end performance. Current SC18 source evaluates raw no-TTA logits, whereas SC12 cached logits follow the upstream flip-TTA evaluation recipe. No temperatures are applied by SC18 source. Saved export alone does not identify every upstream run setting.',data=rep)

def external_figure():
    zero,zp=read_json('SC10','extval_trashnet.json'); tune,tp=read_json('SC10b','trashnet_finetune.json'); taco,dp=read_json('SC10','extval_taco.json')
    fig,ax=plt.subplots(figsize=(5.8,4.2))
    labels=[f'zero-shot: restricted 4-way\n(n={zero["n_images"]})',f'fine-tuned: 6-way\n(n={tune["n_test"]})']
    bars=ax.bar(labels,[zero['acc_restricted_4way'],tune['test_acc']],color=[BLUE,ORANGE],width=.56)
    ax.bar_label(bars,fmt='%.3f',padding=5,fontsize=10); ax.set_ylim(0,1.07)
    ax.set_ylabel('TrashNet accuracy'); ax.grid(axis='y',alpha=.15); ax.set_axisbelow(True)
    fig.subplots_adjust(left=.13,right=.97,bottom=.24,top=.96)
    fig.text(.5,.025,'Different classes and test subsets; these are separate protocols, not a paired gain.',ha='center',fontsize=8,color='#58636B')
    save(fig,'external_validation.png',[zp,tp,dp],f'SC10 classifier: {zero["backbone"]} | SC10b fine-tune init: {tune.get("init","unspecified")} | TACO n={taco["n_images"]}',
         'No implied paired improvement: zero-shot/full, overlap-restricted, and fine-tuned six-way evaluations have incompatible label spaces/subsets. Published literature reference scores are omitted.',data={'zero_shot':zero,'finetuned':tune,'taco':taco})

run_section('Actual end-to-end outcomes',e2e_figure,['end_to_end.png'])
run_section('External validation protocols',external_figure,['external_validation.png'])

## Derive missing SC15 figures from genuine saved view logits; no learned aggregators

In [ ]:
def normalized_entropy(p): return -(p*np.log(p+1e-9)).sum(-1)/np.log(p.shape[-1])

def aggregate_head(method,L,head):
    if head=='condition':
        p=sigmoid(L)
        if method=='mean_logit': return sigmoid(L.mean())
        if method=='mean_prob': return p.mean()
        if method=='median_logit': return sigmoid(np.median(L))
        if method=='best_view': return p[np.argmax(np.abs(p-.5))]
        if method=='max_noisyOR': return 1-np.prod(1-p)
        if method=='conf_weighted':
            w=np.abs(p-.5)+1e-6; return (w*p).sum()/w.sum()
        if method=='lse':
            z=2*L; m=z.max(); return sigmoid((m+np.log(np.exp(z-m).mean()))/2)
    p=corn_probs(L) if head=='threat' else softmax(L)
    decode=corn_probs if head=='threat' else softmax
    if method=='mean_logit': return decode(L.mean(0))
    if method=='mean_prob': return p.mean(0)
    if method=='median_logit': return decode(np.median(L,0))
    if method=='best_view': return p[np.argmax(1-normalized_entropy(p))]
    if method=='conf_weighted':
        w=1-normalized_entropy(p)+1e-6; return (w[:,None]*p).sum(0)/w.sum()
    if method=='lse':
        z=2*L; m=z.max(0); return decode((m+np.log(np.exp(z-m).mean(0)))/2)
    if method=='max_noisyOR':
        if head=='threat': return cumulative_to_probs(1-np.prod(1-corn_cumulative(L),0))
        m=p.max(0); return m/m.sum()
    raise ValueError(method)

def view_aggregation_figure():
    d=diagnostic_data(VIEW_AGGREGATION_SC)
    methods=['mean_logit','mean_prob','conf_weighted','best_view','max_noisyOR','median_logit','lse']; rows=[]
    for method in methods:
        p={h:np.asarray([aggregate_head(method,d['view_logits'][h][ix],h) for ix in d['groups'].values()]) for h in HEADS}
        y=d['labels']; pred=p['threat'].argmax(1); hazard=y['threat']>=2; nonhazard=~hazard
        rows.append({'method':method,'material_f1':f1_score(y['material'],p['material'].argmax(1),labels=range(10),average='macro',zero_division=0),
                     'route_f1':f1_score(y['route'],p['route'].argmax(1),labels=range(4),average='macro',zero_division=0),
                     'threat_qwk':cohen_kappa_score(y['threat'],pred,labels=range(4),weights='quadratic'),
                     'condition_auroc':roc_auc_score(y['condition'],p['condition']),
                     'threat_recall_ge2':float((pred[hazard]>=2).mean()),
                     'threat_false_positive_ge2':float((pred[nonhazard]>=2).mean())})
    df=pd.DataFrame(rows); df.to_csv(OUTPUT/'view_aggregation_derived.csv',index=False)
    # Preserve the original material/threat panels and the genuine two-dimensional
    # safety trade-off. Leader labels share a point when methods truly coincide.
    order=['mean_logit','mean_prob','conf_weighted','best_view','median_logit','lse','max_noisyOR']
    plot=df.set_index('method').loc[order].reset_index(); x=np.arange(len(plot))
    names={'mean_logit':'Mean logits','mean_prob':'Mean probs.','conf_weighted':'Conf. weighted','best_view':'Best view','max_noisyOR':'Max / noisy-OR','median_logit':'Median logits','lse':'Log-sum-exp'}
    fig,axes=plt.subplots(1,2,figsize=(10.4,4.7),gridspec_kw={'width_ratios':[1.2,1]})
    for offset,key,label,color in [(-.18,'material_f1','material macro-F1',BLUE),(.18,'threat_qwk','threat QWK',ORANGE)]:
        axes[0].bar(x+offset,plot[key],width=.34,label=label,color=color,zorder=3)
    axes[0].set_xticks(x,[f'{i+1}. {names[m]}' for i,m in enumerate(order)],rotation=38,ha='right',fontsize=8.5)
    axes[0].set_ylim(0,1.0); axes[0].set_ylabel('item-level score'); axes[0].grid(axis='y',alpha=.15); axes[0].set_axisbelow(True)
    axes[0].legend(loc='upper center',bbox_to_anchor=(.5,1.12),fontsize=9,frameon=False,ncol=2)
    ax=axes[1]; coordinates=defaultdict(list)
    for i,row in plot.iterrows():
        xy=(float(row.threat_false_positive_ge2),float(row.threat_recall_ge2)); coordinates[xy].append(i+1)
        ax.scatter(*xy,s=58,color=TEAL if row.method=='mean_logit' else ORANGE if row.method=='max_noisyOR' else GRAY,zorder=3,edgecolor='white',linewidth=.6)
    xmin=float(plot.threat_false_positive_ge2.min()); xmax=float(plot.threat_false_positive_ge2.max())
    ymin=float(plot.threat_recall_ge2.min()); ymax=float(plot.threat_recall_ge2.max())
    ax.set_xlim(max(-.01,xmin-.045),min(1.02,xmax+max(.14,(xmax-xmin)*.8)))
    ax.set_ylim(max(-.01,ymin-.018),min(1.02,ymax+.018))
    groups=sorted(coordinates.items(),key=lambda pair:(pair[0][1],pair[0][0]))
    for (xy,numbers),slot in zip(groups,np.linspace(.10,.88,len(groups))):
        label=' / '.join(str(n) for n in numbers)
        if len(numbers)>1: label+=' (same point)'
        ax.annotate(label,xy,xytext=(.97,slot),textcoords='axes fraction',ha='right',va='center',fontsize=8.5,
                    bbox=dict(boxstyle='round,pad=.18',fc='white',ec='none',alpha=.95),
                    arrowprops=dict(arrowstyle='-',color='#ADB6BC',lw=.8,shrinkA=3,shrinkB=4))
    ax.set_xlabel('threat-head false-positive rate (true level < 2)')
    ax.set_ylabel('threat-head recall (true level ≥ 2)'); ax.grid(alpha=.15)
    fig.subplots_adjust(left=.07,right=.985,top=.89,bottom=.30,wspace=.29)
    fig.text(.5,.025,'Scatter labels use the method numbers at left. Points retain their measured coordinates; no jitter is applied.',ha='center',fontsize=8,color='#58636B')
    ev=f'Derived from {d.get("label",VIEW_AGGREGATION_SC)} saved logits | {len(d["items"])} items, {d["n_views"]} views'
    save(fig,'view_aggregation_plot.png',[d['path']],ev,
         'Reimplements SC15 seven parameter-free methods. SC15 threat decoding uses CORN class argmax, unlike full_report ordinal threshold decoding. Threat≥2 head recall is not Special-routing hazard recall. Evaluation identity follows the selected verified OOF archive or explicit held-out source. No learned aggregator training or test-based recommendation is made.',status='derived',data=df.to_dict('records'))

run_section('Derivable view aggregation',view_aggregation_figure,['view_aggregation_plot.png'])

## Cost trade-off requires actual measurement, not published or guessed numbers

In [ ]:
def state_tensor_size(sc):
    import torch
    name,_=MODEL_NAMES[sc]
    checkpoint=artifact(sc,f'weights_{name}.pt')
    state=torch.load(str(checkpoint),map_location='cpu',weights_only=True,mmap=True)
    if not isinstance(state,dict) or not all(isinstance(v,torch.Tensor) for v in state.values()):
        raise EvidenceError(f'{sc} checkpoint is not a plain tensor state dictionary.')
    ignored={k:int(v.numel()) for k,v in state.items() if k.endswith(('running_mean','running_var','num_batches_tracked'))}
    other={k:int(v.numel()) for k,v in state.items() if k not in ignored}
    total=sum(other.values()); del state
    return total,checkpoint,ignored

def partial_efficiency_figure():
    reference=model_data('SC12'); fusion=model_data('SC26'); assert_same_cohort(reference,fusion)
    base,_=verified_metric('SC12',reference); plus,_=verified_metric('SC26',fusion)
    a,ap,ab=state_tensor_size('SC12'); b,bp,bb=state_tensor_size('SC26')
    fig,axes=plt.subplots(1,3,figsize=(10.8,3.8),gridspec_kw={'width_ratios':[1,.95,1.5]})
    fig.subplots_adjust(left=.075,right=.975,bottom=.22,top=.85,wspace=.68)
    bars=axes[0].bar(['ConvNeXt-S','Fusion\n(4 backbones)'],[a/1e6,b/1e6],color=[ORANGE,GRAY],width=.58)
    axes[0].bar_label(bars,fmt='%.2f M',padding=4,fontsize=9); axes[0].set_ylim(0,max(a,b)/1e6*1.27)
    axes[0].set_title(f'Size: {b/a:.2f}×',pad=12); axes[0].set_ylabel('learned state tensors (millions)'); axes[0].grid(axis='y',alpha=.15); axes[0].set_axisbelow(True)
    axes[1].set_title('Inference latency',pad=12); axes[1].axis('off')
    axes[1].text(.5,.57,'Not measured',ha='center',va='center',fontsize=13,color='#58636B')
    axes[1].text(.5,.31,'Same hardware, precision,\nbatch and inference recipe\nrequired for comparison.',ha='center',va='center',fontsize=9,color='#58636B',linespacing=1.5)
    keys=['material_f1','threat_qwk','route_f1','condition_auroc']; names=['material F1','threat QWK','route F1','condition AUROC']
    delta=np.array([plus[k]-base[k] for k in keys]); axes[2].barh(names,delta,color=[GREEN if v>=0 else RED for v in delta],height=.55,zorder=3)
    for j,v in enumerate(delta): axes[2].annotate(f'{v:+.4f}',(v,j),xytext=(5 if v>=0 else -5,0),textcoords='offset points',va='center',ha='left' if v>=0 else 'right',fontsize=9)
    axes[2].invert_yaxis(); axes[2].axvline(0,color=GRAY,lw=1)
    axes[2].set_xlim(min(-.003,float(delta.min())-.003),max(.047,float(delta.max())+.016))
    axes[2].set(title='Fusion − ConvNeXt-S',xlabel='held-out score difference'); axes[2].grid(axis='x',alpha=.15); axes[2].set_axisbelow(True)
    fig.text(.5,.025,'Size excludes BatchNorm running statistics/counters. Latency and instantiated-model parameter validation remain pending.',ha='center',fontsize=8,color='#58636B')
    save(fig,'efficiency_tradeoff.png',[ap,bp,reference['path'],fusion['path']],
         f'SC12 vs SC26 | matched {len(reference["items"])}-item test | latency not measured',
         'Partial evidence: counts include state tensors excluding BatchNorm running statistics/counters, not an independently instantiated model parameter count. Includes learned positional/class-token tensors. No latency value or ratio is inferred. Set CLASSIFIER_EFFICIENCY_CSV to a measured matched export to enable the full comparison.',status='partial',
         data={'SC12_state_tensors':a,'SC26_state_tensors':b,'excluded_SC12_buffers':ab,'excluded_SC26_buffers':bb,'SC26_minus_SC12':dict(zip(keys,delta))})
    MANIFEST[-1]['missing_components']=['measured latency','instantiated-model parameter count validation']

def efficiency_figure():
    if not CLASSIFIER_EFFICIENCY_CSV:
        return partial_efficiency_figure()
    path=Path(CLASSIFIER_EFFICIENCY_CSV); data=pd.read_csv(path)
    required=['source','params_M','infer_ms','device','batch_size','input_size','precision','preprocessing_included']
    if any(c not in data for c in required): raise EvidenceError(f'Efficiency CSV needs columns {required}')
    selected=data[data.source.isin(['SC12','SC7','SC26'])].copy()
    if len(selected)<2 or 'SC12' not in selected.source.values or selected.source.duplicated().any(): raise EvidenceError('Need unique SC12 plus one fusion benchmark row.')
    for key in ['device','batch_size','input_size','precision','preprocessing_included']:
        if selected[key].nunique()!=1: raise EvidenceError(f'Incompatible timing conditions: {key}')
    if not np.isfinite(selected[['params_M','infer_ms']].to_numpy(dtype=float)).all() or (selected[['params_M','infer_ms']]<=0).any().any():
        raise EvidenceError('Measured parameter counts and latency must be finite and positive.')
    reference=model_data('SC12'); sources=[path]; metrics={}
    for i,row in selected.iterrows():
        d=model_data(row.source); assert_same_cohort(reference,d); m,_=verified_metric(row.source,d)
        metrics[row.source]=m
        for key,value in m.items(): selected.loc[i,key]=value
        sources.append(d['path'])
    selected=selected.set_index('source').loc[[sc for sc in ['SC12','SC7','SC26'] if sc in selected.source.values]].reset_index()
    focus='SC26' if 'SC26' in metrics else 'SC7'
    fig,axes=plt.subplots(1,3,figsize=(10.8,3.8),gridspec_kw={'width_ratios':[1,1,1.5]})
    fig.subplots_adjust(left=.075,right=.975,bottom=.22,top=.86,wspace=.68)
    for ax,key,label,title in [(axes[0],'params_M','learned parameters (millions)','Model size'),(axes[1],'infer_ms','measured latency (ms / image)','Inference latency')]:
        bars=ax.bar([MODEL_NAMES[sc][1] for sc in selected.source],selected[key],color=[ORANGE if sc=='SC12' else GRAY for sc in selected.source],width=.56)
        ax.bar_label(bars,fmt='%.2f',padding=4,fontsize=9); ax.set_ylim(0,float(selected[key].max())*1.25)
        ax.set(ylabel=label,title=title); ax.grid(axis='y',alpha=.15); ax.set_axisbelow(True)
    keys=['material_f1','threat_qwk','route_f1','condition_auroc']; names=['material F1','threat QWK','route F1','condition AUROC']
    delta=np.array([metrics[focus][k]-metrics['SC12'][k] for k in keys])
    axes[2].barh(names,delta,color=[GREEN if value>=0 else RED for value in delta],height=.55,zorder=3)
    for j,value in enumerate(delta): axes[2].annotate(f'{value:+.4f}',(value,j),xytext=(5 if value>=0 else -5,0),textcoords='offset points',ha='left' if value>=0 else 'right',va='center',fontsize=9)
    axes[2].invert_yaxis(); axes[2].axvline(0,color=GRAY,lw=1); axes[2].grid(axis='x',alpha=.15); axes[2].set_axisbelow(True)
    axes[2].set(xlim=(min(-.003,float(delta.min())-.003),max(.02,float(delta.max())+.016)),xlabel='held-out score difference',title=f'{MODEL_NAMES[focus][1]} − ConvNeXt-S')
    fig.text(.5,.025,f'Matched measurement: {selected.iloc[0].device}; batch {selected.iloc[0].batch_size}; precision {selected.iloc[0].precision}.',ha='center',fontsize=8,color='#58636B')
    save(fig,'efficiency_tradeoff.png',sources,f'Matched test cohort | device: {selected.iloc[0].device}',data={'measured_rows':selected.to_dict('records'),'difference_source':focus,'head_differences':dict(zip(keys,delta))})

run_section('Measured classifier efficiency',efficiency_figure,['efficiency_tradeoff.png'])

## Qualitative and detector rasters: retain actual supplied pixels with explicit provenance

In [ ]:
def retain_raster(sc,source_name,target_name,evaluation,notes):
    p=artifact(sc,source_name); dest=OUTPUT/target_name
    if dest.resolve()==p.resolve(): raise EvidenceError('Refusing to overwrite an input raster')
    shutil.copy2(p,dest)
    with Image.open(p) as im: w,h=im.size
    outputs=[dest]
    if SHOW_FIGURES:
        from IPython.display import display
        with Image.open(p) as im: display(im.copy())
    register(target_name,'legacy_raster',sources=[p],evaluation=evaluation,notes=notes,files=outputs,
             raster_size=[w,h],pixels_retained=True)

def retain_qualitative_showcase():
    """Keep the requested three examples; never copy an unidentified old strip."""
    expected = list(QUALITATIVE_SHOWCASE_ITEMS)
    excluded = list(QUALITATIVE_EXCLUDED_ITEMS)
    if (not expected or len(set(expected)) != len(expected)
            or set(expected).intersection(excluded) or 'item_218' in expected):
        raise EvidenceError('Qualitative showcase IDs must be unique and exclude item_218.')
    source = artifact('SC22', 'qualitative_demo.png')
    metadata_path = artifact('SC22', 'qualitative_manifest.json', required=False)
    destination = OUTPUT/'qualitative_demo.png'
    if destination.resolve() == source.resolve():
        raise EvidenceError('Refusing to overwrite an input raster')
    sources = [source]
    crop_box = None
    with Image.open(source) as image:
        if image.format != 'PNG':
            raise EvidenceError('SC22 qualitative_demo.png must be an actual PNG image.')
        source_size = list(image.size)
        if metadata_path is not None:
            try:
                metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            except (ValueError, UnicodeError) as exc:
                raise EvidenceError('SC22 qualitative_manifest.json is invalid; rerun updated SC22.') from exc
            if not isinstance(metadata, dict):
                raise EvidenceError('SC22 qualitative_manifest.json must contain a metadata object.')
            if (metadata.get('schema_version') != 1
                    or metadata.get('figure_file') != 'qualitative_demo.png'
                    or metadata.get('figure_sha256') != sha256(source)
                    or metadata.get('raster_size') != source_size):
                raise EvidenceError('SC22 qualitative_manifest.json does not identify its PNG; rerun updated SC22.')
            if (metadata.get('selected_item_ids') != expected
                    or metadata.get('excluded_item_ids') != excluded
                    or set(metadata.get('selected_item_ids', [])).intersection(excluded)):
                raise EvidenceError(f'SC22 showcase does not match the requested order {expected}; rerun updated SC22.')
            sources.append(metadata_path)
            raster = image.copy()
            evaluation = 'SC22 saved three-item pipeline illustration; calibration compatibility requires review'
            notes = ('Showcase membership and PNG identity verified against SC22 qualitative_manifest.json. '
                     'Original saved pixels retained. No new inference is performed here. '
                     'Use only SC12-owned temperatures; the historical SC6 calibration belongs to SC7.')
        else:
            # The supplied four-row strip was inspected visually. Its order is
            # 218, 001, 101, 049; y=528 lies in an all-white separator before 001.
            # This exact content hash prevents applying those coordinates to a
            # differently ordered or regenerated raster. No CSV order is assumed.
            known_sha = 'aea523918b3834058187a16348e50168cdbaa23e59d466f70fb811a81ac2e54a'
            if (sha256(source) != known_sha or source_size != [1749, 2179]
                    or expected != ['item_001', 'item_101', 'item_049']):
                raise EvidenceError('SC22 PNG has no verifiable showcase metadata. Rerun updated SC22 and attach qualitative_manifest.json with its three-item PNG.')
            crop_box = [0, 528, 1749, 2179]
            raster = image.crop(tuple(crop_box))
            evaluation = 'SC22 historical pipeline illustration; item_218 row removed, remaining three examples retained'
            notes = ('Removed only the item_218 row from the exact SHA-256 verified supplied four-row raster. '
                     'The remaining item_001, item_101 and item_049 pixels and labels are unchanged; '
                     'no resizing, redrawing or new inference. Historical SC22 applies SC7-owned SC6 temperatures '
                     'to SC12, so this remains review-only evidence of confidence; rerun corrected SC22 for compatible calibration.')
        output_size = list(raster.size)
    if crop_box is None:
        shutil.copy2(source, destination)
    else:
        raster.save(destination, format='PNG', dpi=(200, 200))
    if SHOW_FIGURES:
        from IPython.display import display
        display(raster)
    register('qualitative_demo.png', 'legacy_raster', sources=sources,
             evaluation=evaluation, notes=notes, files=[destination],
             raster_size=output_size, source_raster_size=source_size,
             pixels_retained=True, showcase_items=expected, excluded_items=excluded,
             source_raster_crop_box=crop_box, showcase_verified=True)
    raster.close()


def qualitative_rasters():
    try:
        retain_qualitative_showcase()
    except (FileNotFoundError, EvidenceError, OSError) as exc:
        print(f'qualitative_demo.png: unavailable: {exc}')
        unavailable(['qualitative_demo.png'], str(exc),
                    'Run updated SC22_qualitative_demo.ipynb and attach its PNG plus qualitative_manifest.json. '
                    'For differing attached versions, set SC_ROOT_OVERRIDES["SC22"].')
    for sc,source,target,ev,note in [
        ('SC21','gradcam_grid.png','gradcam_grid.png','SC21 saved ConvNeXt-S gradient-based CAM output; release identity unverified',
         'Source notebook computes real activation gradients. Raw input/CAM arrays are not exported. This is the original raster, not a redesigned or freshly recomputed attribution. Captions must mention material and threat CAM columns.'),
        ('SC30','fig1_labelling_transparent.png','fig1_labelling_transparent.png','SC30 saved item_106 annotation illustration',
         'Saved output depicts item_106; current source selects item_115. Labels are an illustrative existing export; attach image manifest to regenerate.'),
        ('SC30','fig2_preprocessing_transparent.png','fig2_preprocessing_transparent.png','SC30 saved item_106 preprocessing illustration',
         'Saved output/source item selection differ. These are examples, not measurement evidence. Raw photographs are required to redesign.'),
        ('SC30','fig3_detector_classifier_transparent.png','fig3_detector_classifier_transparent.png','SC30 saved item_106 detect/classify illustration',
         'Retained existing raster. Recompute with a verified detector checkpoint, image manifest and matching classifier; no GT-box fallback should be described as detection.'),
    ]:
        try: retain_raster(sc,source,target,ev,note)
        except (FileNotFoundError,EvidenceError) as exc:
            print(f'{target}: unavailable: {exc}')
            unavailable([target],str(exc),f'Attach {sc} outputs; if versions differ, set SC_ROOT_OVERRIDES[{sc!r}] to the intended producer/version directory. Otherwise rerun its corrected notebook on matching images/checkpoints.')

def detector_rasters():
    for name in ['detection_PR.png','detection_F1.png','detection_P.png','detection_R.png']:
        try:
            retain_raster('Report_Figures',name,name,'Historical SC8 detector evaluation — separate from SC27 485-item reevaluation',
                          'Raw threshold/precision/recall arrays are absent. Existing Results/Report_Figures pixels are retained with their original evaluation provenance. These curves must not be captioned as SC27 curves. Use optional detector reevaluation cell to produce genuine current curves.')
        except (FileNotFoundError,EvidenceError) as exc:
            print(f'{name}: unavailable: {exc}')
            unavailable([name],str(exc),'For conflicting saved versions, set SC_ROOT_OVERRIDES["Report_Figures"] to the intended producer/version directory. Otherwise run optional genuine detector reevaluation with the SC8 best.pt checkpoint and a dataset YAML defining the current test split; never substitute TACO curves.')

qualitative_rasters()
detector_rasters()

## Optional real detector reevaluation, with explicit local artifacts and no download

In [ ]:
def reevaluate_detector():
    if not RUN_DETECTOR_REEVALUATION:
        print('Optional detector reevaluation is disabled. No checkpoint/data is downloaded.')
        return
    if not DETECTOR_CHECKPOINT or not DETECTOR_DATA_YAML:
        raise EvidenceError('Set DETECTOR_CHECKPOINT and DETECTOR_DATA_YAML explicitly. No first best.pt selection.')
    checkpoint=Path(DETECTOR_CHECKPOINT).resolve(); yaml=Path(DETECTOR_DATA_YAML).resolve()
    if not checkpoint.is_file() or not yaml.is_file(): raise FileNotFoundError('Detector checkpoint or data YAML missing')
    import yaml as pyyaml
    dataset=pyyaml.safe_load(yaml.read_text())
    if 'test' not in dataset: raise EvidenceError('Detector YAML must explicitly declare a test split.')
    from ultralytics import YOLO
    model=YOLO(str(checkpoint))
    if len(model.names)!=1: raise EvidenceError('Checkpoint is not a single-class waste detector.')
    evaluation_dir=OUTPUT/'detector_reevaluation'
    result=model.val(data=str(yaml),split='test',conf=.001,iou=.7,imgsz=640,plots=True,
                     project=str(evaluation_dir),name='current_test',exist_ok=False,device=DETECTOR_DEVICE,verbose=False)
    # Ultralytics exposes the real threshold curves in curves_results; no analytic approximations.
    curves=list(getattr(result,'curves_results',[])); titles=list(getattr(result,'curves',[]))
    wanted={'Precision-Recall(B)':'detection_PR.png','F1-Confidence(B)':'detection_F1.png',
            'Precision-Confidence(B)':'detection_P.png','Recall-Confidence(B)':'detection_R.png'}
    exported={}
    for title,entry in zip(titles,curves):
        target=wanted.get(title)
        if target is None: continue
        x,y,xlabel,ylabel=entry; x=np.asarray(x); y=np.asarray(y)
        if y.ndim==1: y=y[None]
        fig,ax=plt.subplots(figsize=(6.0,4.8))
        for k,line in enumerate(y): ax.plot(x,line,color=BLUE,lw=2.4,label=f'Waste item · class {k}')
        ax.set(xlabel=xlabel,ylabel=ylabel,xlim=(0,1),ylim=(0,1.03)); ax.set_title(title.replace('(B)',''),loc='left',pad=14); ax.grid(alpha=.15); ax.legend(fontsize=8)
        MANIFEST[:]=[e for e in MANIFEST if e['figure']!=target]
        save(fig,target,[checkpoint,yaml],'Fresh detector reevaluation | explicitly configured test dataset',
             'Curve coordinates come directly from Ultralytics metrics. Replaces legacy raster entry. Check test item/view counts against the intended release before claiming SC27 reproduction.',status='derived',data={'x':x,'y':y,'xlabel':xlabel,'ylabel':ylabel})
        exported[target]=True
    missing=set(wanted.values())-set(exported)
    if missing:
        print('Ultralytics API did not expose these raw curves:',sorted(missing))
        print('Keep generated framework plots in',result.save_dir,'and export raw threshold curves from the installed version before redesigning them.')
    (OUTPUT/'detector_reevaluation_metrics.json').write_text(json.dumps({
        'checkpoint':str(checkpoint),'checkpoint_sha256':sha256(checkpoint),
        'data_yaml':str(yaml),'yaml_sha256':sha256(yaml),
        'metrics':{k:float(v) for k,v in result.results_dict.items()},
        'save_dir':str(result.save_dir)},indent=2),encoding='utf-8')

reevaluate_detector()

## Complete report reference coverage and reproducibility manifest

In [ ]:
covered={e['figure'] for e in MANIFEST}
if covered-set(REPORT_REFERENCES):
    raise EvidenceError('A generator exported an out-of-scope report figure: '+str(sorted(covered-set(REPORT_REFERENCES))))
for name in REPORT_REFERENCES:
    if name not in covered:
        unavailable([name],'No verified generator or eligible artifact was available for this report reference.',
                    'Attach the missing producer artifacts or raw evidence. No placeholder figure is created.')
manifest={'schema_version':1,'input_roots':list(map(str,INPUT_BASES)),
          'configuration':{'primary_sc':PRIMARY_SC,'diagnostic_sc':DIAGNOSTIC_SC,
                           'prefer_direct_notebook_inputs':PREFER_DIRECT_NOTEBOOK_INPUTS,
                           'cv_source':CV_SOURCE,'include_baseline_fusion':INCLUDE_BASELINE_FUSION,
                           'view_aggregation_sc':VIEW_AGGREGATION_SC,
                           'qualitative_showcase_items':QUALITATIVE_SHOWCASE_ITEMS,
                           'qualitative_excluded_items':QUALITATIVE_EXCLUDED_ITEMS,
                           'sc_root_overrides':SC_ROOT_OVERRIDES,
                           'run_detector_reevaluation':RUN_DETECTOR_REEVALUATION},
          'method':{'item_aggregation':'float64 sum/mean per physical item, then float32',
                    'threat_metrics_decoder':'sum(cumprod(sigmoid(logits)) > 0.5)',
                    'sc15_threat_decoder':'argmax(CORN class probabilities)',
                    'ece':'15 equal-width bins, (lower,upper], class-argmax accuracy',
                    'formats':'PNG only; new charts rendered at 600 dpi; retained rasters preserve original pixels'},
          'report_references':REPORT_REFERENCES,'figures':MANIFEST,'discovery_events':SOURCE_EVENTS,
          'all_report_references':ALL_REPORT_REFERENCES,'excluded_report_figures':EXCLUDED_REPORT_FIGURES,
          'style_reference_assets':STYLE_REFERENCES,
          'resolved_producer_roots':{k:str(v) for k,v in RESOLVED.items()},
          'software_versions':{'numpy':np.__version__,'pandas':pd.__version__,'matplotlib':__import__('matplotlib').__version__,
                               'scikit_learn':__import__('sklearn').__version__}}
(OUTPUT/'generation_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
table=pd.DataFrame([{k:e[k] for k in ['figure','status','evaluation','notes']} for e in MANIFEST])
table.to_csv(OUTPUT/'figure_status.csv',index=False)
pd.DataFrame(EXCLUDED_REPORT_FIGURES).to_csv(OUTPUT/'excluded_report_figures.csv',index=False)
readme=['# Generated report figures','',
        'Report is unchanged. Every input used is hashed in generation_manifest.json.',
        'PNG is the only figure format. New charts use 600 dpi and retain the report basenames.',
        'legacy_raster means supplied PNG pixels were retained; they were not redrawn.',
        'partial means only documented evidence components are available; no timing claim may be made from a size-only efficiency figure.',
        'Previous exports are preserved in a sibling output-history folder; PDF/SVG and obsolete diagrams are outside the current deliverable.',
        'Different release populations remain explicitly labeled. No fake values, curves or missing-data placeholders are created.','']
for entry in MANIFEST:
    if entry['status'] in ['missing','partial','legacy_raster'] or entry.get('missing_components'):
        readme.extend([f'## {entry["figure"]} [{entry["status"]}]',entry['notes'],entry.get('action',''),
                       'Pending: '+', '.join(entry.get('missing_components',[])) if entry.get('missing_components') else '',''])
readme.extend(['## Manually maintained report diagrams','',
               'These assets are intentionally excluded from notebook generation. Their report originals and Draw.io sources remain unchanged.',''])
for entry in EXCLUDED_REPORT_FIGURES:
    readme.append(f'- {entry["figure"]}: {entry["reason"]}')
(OUTPUT/'README.md').write_text('\n'.join(readme),encoding='utf-8')
print(table[['figure','status']].to_string(index=False))
print('\nStatus counts:',table.status.value_counts().to_dict())
print('Manifest:',OUTPUT/'generation_manifest.json')
if FAIL_ON_MISSING and (table.status=='missing').any():
    raise EvidenceError('Some report figures need additional evidence; see figure_status.csv and README.md. Available figures were generated successfully.')

## Export and caption review

`generation_manifest.json` is the definitive record of what was generated, derived, retained as an existing raster, or left missing. `figure_status.csv` is a compact review list. For every reused basename, read the `evaluation` and `notes` fields before retaining the original report caption.

For the three-row qualitative showcase, run the updated `SC22_qualitative_demo.ipynb` first and attach its output to this notebook. Keep `qualitative_demo.png` and `qualitative_manifest.json` together. `pipeline_predictions.csv` still contains all successfully evaluated test items, including item_218; exclusion applies to the illustration only. The verified historical four-row input remains supported by removing its first row without altering the retained examples.

To create genuine new Grad-CAM or photography panels with a better layout, the upstream notebook must additionally export original crops, raw CAM arrays, exact item/view IDs, predicted boxes, checkpoint hashes, temperatures with their owner, and ground-truth labels. Pixels in an existing composed PNG cannot establish those missing facts. Rerun the corrected SC21/SC22/SC30 producer with those exports, then design from the actual arrays.

For classifier efficiency, instantiate the exact architecture with `pretrained=False`, load the explicit checkpoint, compute `sum(p.numel() for p in model.parameters())`, and time matched inference under `model.eval()` and `torch.inference_mode()`. Use at least 30 warmup iterations and 100 measured iterations, call `torch.cuda.synchronize()` before and after each GPU timing window, and report median milliseconds per image with hardware, precision, input shape, TTA/view aggregation, batch size and preprocessing scope. Export the measured CSV described in the configuration cell. Tensor elements in a saved state dictionary include buffers and must not be relabeled as strictly learned parameters.
